<div style="
    background: linear-gradient(135deg, #102542 0%, #0F4C5C 45%, #1A759F 100%);
    padding: 34px;
    border-radius: 14px;
    margin: 10px 0 20px 0;
    box-shadow: 0 10px 26px rgba(16,37,66,0.34);
    border: 1px solid rgba(255,255,255,0.12);
">
    <h1 style="
        color: #ffffff;
        margin: 0;
        text-align: center;
        font-size: 34px;
        font-weight: 300;
        letter-spacing: 2px;
    ">PHY-3500 · Notebook 02</h1>
    <p style="
        color: rgba(255,255,255,0.92);
        margin: 10px 0 0 0;
        text-align: center;
        font-size: 18px;
        letter-spacing: 0.5px;
    ">UMAP et t-SNE sur les scores PCA</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.14), rgba(26,117,159,0.12));
    border: 1px solid rgba(15,76,92,0.30);
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<strong>Équipe :</strong> Alex, Justine<br>
<strong>Cours :</strong> PHY-3500 Physique numérique - Université Laval<br>
<strong>Prérequis :</strong> Notebook 01 (01_pca.ipynb) exécuté pour charger les scores PCA
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.14), rgba(16,37,66,0.14));
    border: 1px solid rgba(26,117,159,0.30);
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<h3 style="margin: 0 0 10px 0; color: #0F4C5C;">Objectifs</h3>
<ol style="margin: 0; padding-left: 20px; line-height: 1.6;">
  <li>Calculer des embeddings UMAP et t-SNE à partir des scores PCA.</li>
  <li>Visualiser la structure 2D avec colorations physiques (T_eff, log g, [Fe/H], BP-RP).</li>
  <li>Évaluer la stabilité via Procrustes multi-seeds.</li>
  <li>Vérifier un contrôle négatif sur des données permutées.</li>
  <li>Explorer la sensibilité aux hyperparamètres.</li>
  <li>Projeter les embeddings sur le diagramme HR.</li>
</ol>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 14px 16px;
    margin-top: 12px;
">
<strong>Note méthodologique :</strong> UMAP/t-SNE sont appliqués sur les scores PCA (et non sur X brut) pour réduire le bruit, accélérer le calcul et stabiliser la topologie des embeddings.
</div>

<div style="
    background: linear-gradient(135deg, rgba(16,37,66,0.10), rgba(26,117,159,0.10));
    border-left: 5px solid #1A759F;
    border-radius: 8px;
    padding: 10px 12px;
    margin-top: 10px;
">
<strong>Lecture scientifique :</strong> ce notebook compare deux méthodes non linéaires complémentaires.
UMAP préserve mieux les structures globales du graphe de voisinage, tandis que t-SNE optimise surtout la cohérence locale.
</div>

<a id="toc"></a>

## Table des matières

<div style="display: grid; grid-template-columns: repeat(2, minmax(260px, 1fr)); gap: 12px; margin: 14px 0 22px 0;">

  <div style="background: linear-gradient(135deg, #0F4C5C, #1A759F); padding: 14px; border-radius: 10px;">
    <a href="#setup" style="color: white; text-decoration: none;"><strong>0. Imports et chargement</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #1A759F, #0F4C5C); padding: 14px; border-radius: 10px;">
    <a href="#umap" style="color: white; text-decoration: none;"><strong>1. Embedding UMAP</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #0F4C5C, #134B5F); padding: 14px; border-radius: 10px;">
    <a href="#tsne" style="color: white; text-decoration: none;"><strong>2. Embedding t-SNE</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #134B5F, #1A759F); padding: 14px; border-radius: 10px;">
    <a href="#stability" style="color: white; text-decoration: none;"><strong>3. Stabilité (Procrustes)</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #1A759F, #0F4C5C); padding: 14px; border-radius: 10px;">
    <a href="#negative" style="color: white; text-decoration: none;"><strong>4. Contrôle négatif</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #0F4C5C, #134B5F); padding: 14px; border-radius: 10px;">
    <a href="#sensitivity" style="color: white; text-decoration: none;"><strong>5. Sensibilité hyperparamètres</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #134B5F, #1A759F); padding: 14px; border-radius: 10px;">
    <a href="#hr" style="color: white; text-decoration: none;"><strong>6. Diagramme HR</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #1A759F, #0F4C5C); padding: 14px; border-radius: 10px;">
    <a href="#save" style="color: white; text-decoration: none;"><strong>7. Sauvegarde des embeddings</strong></a>
  </div>

</div>

<a id="setup"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">0 · Imports et chargement des scores PCA</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Cette cellule initialise l’environnement, charge les artefacts du NB01 et fixe les chemins de sortie.
On garantit ainsi une reproductibilité stricte (même dataset, même scores PCA, même seed).
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# ── Environnement projet (même pattern que 00_master_pipeline) ─────────────
sys.path.insert(0, str(Path("../..").resolve() / "src"))

from utils import setup_project_env

paths = setup_project_env(verbose=True)

# ── Chemins ────────────────────────────────────────────────────────────────
PCA_OUTPUT  = Path(paths["REPORTS_DIR"]) / "phy3500_pca_output.joblib"

if not PCA_OUTPUT.exists():
    raise FileNotFoundError(
        f"Fichier PCA introuvable : {PCA_OUTPUT}\n"
        "→ Lancer d'abord 01_pca.ipynb (cellule 11 — Sauvegarde)."
    )

# ── Imports dimred ──────────────────────────────────────────────────────────
from dimred import EmbeddingEngine, DimRedVisualizer
from dimred.embedding import compare_embeddings
from dimred.dimred_visualizer import CLASS_COLORS, CLASS_MARKERS

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
RANDOM_STATE = 42

# ── Chargement des scores PCA ──────────────────────────────────────────────
pca_output  = joblib.load(PCA_OUTPUT)
scores_95   = pca_output["scores_95pct"]   # Features (95% variance)
y           = pca_output["y"]
meta        = pca_output["meta"]
n_pcs       = pca_output["n_components_95pct"]

# ── Dossier figures (sous-dossier par fichier de features) ─────────────────
FIGURES_ROOT = Path(paths["REPORTS_DIR"]) / "figures" / "phy3500"
features_stem = pca_output.get("features_stem", "unknown")
FIGURES_DIR   = FIGURES_ROOT / features_stem
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nScores PCA chargés : {pca_output['scores'].shape}")
print(f"Scores 95% ({n_pcs} PCs) : {scores_95.shape}")
print(f"Classes : {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Figures : {FIGURES_DIR}")

viz = DimRedVisualizer(figsize=(9, 7), dpi=150, output_dir=FIGURES_DIR)

<a id="umap"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #134B5F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1 · Embedding UMAP</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<strong>Principe :</strong> UMAP construit un graphe de voisinage local (k-NN), puis cherche une projection 2D qui préserve au mieux la structure de connectivité probabiliste.
</div>

La fonction objectif de UMAP peut être vue comme une cross-entropie entre les similarités haute dimension et basse dimension :

$$
\mathcal{L}_{\mathrm{UMAP}} = \sum_{i,j}\left[w_{ij}\log\left(\frac{w_{ij}}{\tilde w_{ij}}\right) + (1-w_{ij})\log\left(\frac{1-w_{ij}}{1-\tilde w_{ij}}\right)\right]
$$

- `n_neighbors` contrôle l’échelle locale de la géométrie (petit = très local, grand = plus global).
- `min_dist` contrôle la compacité des amas en 2D (petit = groupes plus serrés).

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #1A759F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
umap_engine = EmbeddingEngine(
    method="umap",
    n_components=2,
    random_state=RANDOM_STATE,
    n_neighbors=15,
    min_dist=0.1,
)

Z_umap = umap_engine.fit_transform(scores_95)
print(f"UMAP terminé en {umap_engine.fit_time_:.1f}s | shape : {Z_umap.shape}")

In [ ]:
# Coloration par classe
fig, ax = viz.plot_embedding(
    Z_umap, y=y,
    title="UMAP — Spectres LAMOST DR5 (n_neighbors=15, min_dist=0.1)",
    method="UMAP", s=2, alpha=0.45,
    save_path=FIGURES_DIR / "umap_classes.png",
)
plt.show()

In [ ]:
# ============================================================================
# UMAP zoom + comparaison de classes (2 paires de figures, chaque paire ensemble)
# Pair 1: STAR + GALAXY + QSO (full + zoom dans UNE meme figure)
# Pair 2: GALAXY + QSO (STAR filtree) (full + zoom dans UNE meme figure)
# ============================================================================

# >>> MODIFIER ICI : FENETRE DE ZOOM <<<
ZOOM_X = (1.0, 4.0)  # ex: (2.5, 4.5)
ZOOM_Y = (5.5, 8.0)  # ex: (2.0, 5.0)

x_min, x_max = ZOOM_X
y_min, y_max = ZOOM_Y
if not (x_min < x_max and y_min < y_max):
    raise ValueError("La fenetre de zoom est invalide: verifier ZOOM_X et ZOOM_Y.")

# Normaliser les etiquettes de classes (robuste aux variantes/typos)
labels = np.char.upper(np.asarray(y).astype(str))
labels = np.char.strip(labels)
label_aliases = {
    "GLAXAXY": "GALAXY",
    "GALAXIE": "GALAXY",
    "QS0": "QSO",
}
for bad, good in label_aliases.items():
    labels = np.where(labels == bad, good, labels)

# Styles visuels par classe
styles = {
    "STAR": {
        "color": CLASS_COLORS.get("STAR", "#4C72B0"),
        "marker": CLASS_MARKERS.get("STAR", "o"),
        "size_full": 2,
        "alpha_full": 0.22,
        "size_zoom": 6,
        "alpha_zoom": 0.35,
    },
    "GALAXY": {
        "color": CLASS_COLORS.get("GALAXY", "#DD8452"),
        "marker": CLASS_MARKERS.get("GALAXY", "s"),
        "size_full": 18,
        "alpha_full": 0.85,
        "size_zoom": 42,
        "alpha_zoom": 0.95,
    },
    "QSO": {
        "color": CLASS_COLORS.get("QSO", "#55A868"),
        "marker": CLASS_MARKERS.get("QSO", "^"),
        "size_full": 22,
        "alpha_full": 0.90,
        "size_zoom": 48,
        "alpha_zoom": 0.95,
    },
}

from matplotlib.patches import Rectangle


def plot_pair_together(class_list, base_name, title_prefix):
    """Genere une figure 1x2: panneau gauche=vue complete, droite=zoom."""
    zoom_window_mask = (
        (Z_umap[:, 0] >= x_min) & (Z_umap[:, 0] <= x_max)
        & (Z_umap[:, 1] >= y_min) & (Z_umap[:, 1] <= y_max)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

    # ----- Panneau gauche : vue complete -----
    ax = axes[0]
    for cls in class_list:
        cls_mask = labels == cls
        n_cls = int(cls_mask.sum())
        if n_cls == 0:
            continue

        st = styles[cls]
        ax.scatter(
            Z_umap[cls_mask, 0],
            Z_umap[cls_mask, 1],
            c=st["color"],
            marker=st["marker"],
            s=st["size_full"],
            alpha=st["alpha_full"],
            linewidths=0,
            rasterized=True,
            label=f"{cls} (n={n_cls})",
        )

    ax.add_patch(
        Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            fill=False,
            edgecolor="#2F3E46",
            linewidth=1.8,
            linestyle="--",
        )
    )

    ax.set_title(f"{title_prefix} - vue complete", fontsize=11)
    ax.set_xlabel("UMAP axis 1")
    ax.set_ylabel("UMAP axis 2")
    ax.grid(True, alpha=0.25)

    handles_full, _ = ax.get_legend_handles_labels()
    if handles_full:
        ax.legend(framealpha=0.92, loc="best")

    # ----- Panneau droit : zoom -----
    ax = axes[1]
    total_zoom_points = 0

    for cls in class_list:
        cls_zoom_mask = (labels == cls) & zoom_window_mask
        n_zoom = int(cls_zoom_mask.sum())
        if n_zoom == 0:
            continue

        total_zoom_points += n_zoom
        st = styles[cls]
        ax.scatter(
            Z_umap[cls_zoom_mask, 0],
            Z_umap[cls_zoom_mask, 1],
            c=st["color"],
            marker=st["marker"],
            s=st["size_zoom"],
            alpha=st["alpha_zoom"],
            linewidths=0,
            label=f"{cls} (n={n_zoom})",
        )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(
        f"{title_prefix} - zoom x=[{x_min},{x_max}], y=[{y_min},{y_max}]",
        fontsize=11,
    )
    ax.set_xlabel("UMAP axis 1")
    ax.set_ylabel("UMAP axis 2")
    ax.grid(True, alpha=0.25)

    handles_zoom, _ = ax.get_legend_handles_labels()
    if handles_zoom:
        ax.legend(framealpha=0.92, loc="best")
    else:
        ax.text(
            0.5,
            0.5,
            "Aucun point pour ces classes\ndans cette fenetre de zoom",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#999999", alpha=0.9),
        )

    fig.suptitle(
        f"{title_prefix} | Comparaison vue complete vs zoom",
        fontsize=13,
        fontweight="bold",
        y=1.02,
    )
    fig.tight_layout()

    pair_path = FIGURES_DIR / f"{base_name}_pair.png"
    fig.savefig(pair_path, bbox_inches="tight", dpi=150)
    plt.show()

    print(f"Saved: {pair_path}")
    for cls in class_list:
        n_zoom = int(((labels == cls) & zoom_window_mask).sum())
        print(f"  {cls} in zoom: {n_zoom}")
    print(f"  Total in zoom: {total_zoom_points}")


# Pair 1: STAR + GALAXY + QSO ensemble (figures 1 et 2)
plot_pair_together(
    class_list=("STAR", "GALAXY", "QSO"),
    base_name="umap_all_classes",
    title_prefix="UMAP STAR + GALAXY + QSO",
)

# Pair 2: GALAXY + QSO ensemble (figures 3 et 4)
plot_pair_together(
    class_list=("GALAXY", "QSO"),
    base_name="umap_no_star",
    title_prefix="UMAP GALAXY + QSO (STAR filtree)",
)

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Interprétation - Structure UMAP et non-linéarité</h4>
<p style="margin: 0 0 8px 0;">L’embedding UMAP révèle des filaments et des sous-populations absents d’une projection linéaire simple. Cela indique que la variété spectrale est intrinsèquement non linéaire.</p>
<p style="margin: 0;">Une continuité de gradient en T_eff et [Fe/H] à travers ces structures suggère que la projection conserve une information astrophysique réelle et pas seulement une séparation artificielle.</p>
</div>

### Lecture scientifique rapide
- UMAP préserve bien les voisinages locaux tout en gardant une structure globale exploitable.
- La présence d’amas denses et de transitions progressives est cohérente avec des populations stellaires en continuum physique.
- Une comparaison avec le contrôle négatif est essentielle pour valider que cette géométrie n’est pas un artefact de la méthode.

In [ ]:
# Grille multi-coloration
fig, axes = viz.plot_embedding_grid(
    Z_umap, y=y, meta=meta,
    method="UMAP",
    params=["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"],
    s=2, alpha=0.4,
    save_path=FIGURES_DIR / "umap_grid.png",
)
plt.show()

<a id="hdbscan"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.2 · Clustering automatisé — HDBSCAN sur espace UMAP</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Objectif</h4>
<p style="margin: 0 0 8px 0;">
UMAP révèle une structure filamentaire riche — mais sans étiquettes automatiques. On applique <strong>HDBSCAN</strong> (Hierarchical Density-Based Spatial Clustering of Applications with Noise) directement sur les coordonnées 2D de UMAP pour <em>quantifier mathématiquement</em> les populations stellaires identifiées visuellement.
</p>
<p style="margin: 0;">
Avantage clé de HDBSCAN vs K-Means : il gère les formes arbitraires (filaments, croissants), ne force pas tous les points dans un cluster, et classe les objets atypiques comme <em>bruit</em> — ce qui en astrophysique correspond aux anomalies spectrales (QSOs, galaxies, spectres défectueux).
</p>
</div>

In [ ]:
# ── Import HDBSCAN (avec message d'installation si absent) ──────────────
try:
    import hdbscan
    _HDBSCAN_OK = True
except ImportError:
    print("⚠  HDBSCAN non installé → pip install hdbscan")
    _HDBSCAN_OK = False

if not _HDBSCAN_OK:
    raise RuntimeError("Installer hdbscan avant de continuer.")

# ── Paramètres HDBSCAN ───────────────────────────────────────────────────
# min_cluster_size : taille minimale d'un amas pour être retenu
#   → 150 sur 42 920 spectres ≈ 0.35% du jeu — détecte les sous-populations -> pas vraiment....
#     significatives sans fragmenter la séquence principale
# min_samples      : nombre de voisins pour qu'un point soit considéré
#                    comme cœur de densité; plus élevé = plus de bruit classifié
HDBSCAN_MIN_CLUSTER = 75
HDBSCAN_MIN_SAMPLES = 20# ── Import HDBSCAN ──────────────────────────────────────────────────────────
try:
    import hdbscan
    _HDBSCAN_OK = True
except ImportError:
    print("⚠  HDBSCAN non installé → pip install hdbscan")
    _HDBSCAN_OK = False

if not _HDBSCAN_OK:
    raise RuntimeError("Installer hdbscan avant de continuer.")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

# ── Paramètres HDBSCAN ───────────────────────────────────────────────────────
# min_cluster_size=75 : granularité fine, ~29 clusters, 9.3% bruit
# Validé empiriquement sur les données LAMOST DR5 × Gaia DR3 (42 920 spectres)
HDBSCAN_MIN_CLUSTER = 75
HDBSCAN_MIN_SAMPLES = 20

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

cluster_labels = clusterer.fit_predict(Z_umap)

n_clusters  = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise     = (cluster_labels == -1).sum()
n_clustered = (cluster_labels != -1).sum()
cluster_ids = sorted(set(cluster_labels) - {-1})

print(f"HDBSCAN terminé :")
print(f"  Clusters détectés    : {n_clusters}")
print(f"  Points classifiés    : {n_clustered} ({100*n_clustered/len(cluster_labels):.1f}%)")
print(f"  Points bruit (noise) : {n_noise} ({100*n_noise/len(cluster_labels):.1f}%)")

# Composition par classe LAMOST
df_hdb = pd.DataFrame({'cluster': cluster_labels, 'class_lamost': y})
cross = pd.crosstab(df_hdb['cluster'], df_hdb['class_lamost'])
cross.index.name = 'Cluster'
print("\nComposition par classe LAMOST :")
print(cross.to_string())

# ── Palette de couleurs ──────────────────────────────────────────────────────
cmap_clusters = plt.get_cmap('turbo', n_clusters)
color_map = {cid: cmap_clusters(i / max(1, n_clusters - 1))
             for i, cid in enumerate(cluster_ids)}
color_map[-1] = (0.82, 0.82, 0.82, 0.25)

noise_mask = cluster_labels == -1

# ── Helper : centroïde d'un cluster dans un espace 2D ───────────────────────
def centroid(Z, mask):
    return Z[mask].mean(axis=0)

# ── Helper : annotation compacte sur chaque cluster ─────────────────────────
def annotate_clusters(ax, Z, labels, ids, threshold_label=0):
    """Trace le numéro de cluster sur son centroïde si n > threshold_label."""
    for cid in ids:
        mask = labels == cid
        n    = mask.sum()
        if n < threshold_label:
            continue
        cx, cy = centroid(Z, mask)
        ax.text(
            cx, cy, f"C{cid}",
            fontsize=7, fontweight='bold',
            ha='center', va='center',
            color='white',
            bbox=dict(boxstyle='round,pad=0.15', fc=color_map[cid],
                      ec='none', alpha=0.85),
        )

# ═════════════════════════════════════════════════════════════════════════════
# Figure 1 — Clusters HDBSCAN vs Classes LAMOST (avec annotations centroïdes)
# ═════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=150)

# ── Panneau gauche : clusters HDBSCAN annotés ────────────────────────────────
ax = axes[0]
ax.scatter(
    Z_umap[noise_mask, 0], Z_umap[noise_mask, 1],
    c=[color_map[-1]], s=1.5, alpha=0.25,
    rasterized=True, linewidths=0, zorder=1,
)
for cid in cluster_ids:
    mask = cluster_labels == cid
    ax.scatter(
        Z_umap[mask, 0], Z_umap[mask, 1],
        c=[color_map[cid]], s=2, alpha=0.75,
        rasterized=True, linewidths=0, zorder=2,
    )

# Annotations texte sur TOUS les clusters (seuil 0 = annoter tout)
annotate_clusters(ax, Z_umap, cluster_labels, cluster_ids, threshold_label=0)

# Colorbar discrète à la place de la légende
sm = plt.cm.ScalarMappable(
    cmap=cmap_clusters,
    norm=mcolors.Normalize(vmin=0, vmax=n_clusters - 1),
)
sm.set_array([])
cb = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, label='Cluster ID')
cb.ax.tick_params(labelsize=8)

ax.text(
    0.02, 0.98, f"Bruit : n={n_noise} ({100*n_noise/len(cluster_labels):.1f}%)",
    transform=ax.transAxes, fontsize=8, va='top',
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.85),
)
ax.set_xlabel('UMAP axe 1', fontsize=11)
ax.set_ylabel('UMAP axe 2', fontsize=11)
ax.set_title(f'HDBSCAN — {n_clusters} populations identifiées', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)

# ── Panneau droit : classes LAMOST pour comparaison ──────────────────────────
ax = axes[1]
for cls in np.unique(y):
    mask = y == cls
    ax.scatter(
        Z_umap[mask, 0], Z_umap[mask, 1],
        c=CLASS_COLORS.get(cls, '#888888'),
        marker=CLASS_MARKERS.get(cls, 'o'),
        s=2, alpha=0.45, label=f'{cls} (n={mask.sum()})',
        rasterized=True, linewidths=0,
    )
ax.set_xlabel('UMAP axe 1', fontsize=11)
ax.set_title('Classes LAMOST (référence)', fontsize=13, fontweight='bold')
ax.legend(markerscale=4, fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.2)

fig.suptitle(
    f'HDBSCAN sur UMAP — {n_clusters} clusters · '
    f'{n_noise} points bruit ({100*n_noise/len(cluster_labels):.1f}%)\n'
    'LAMOST DR5 × Gaia DR3',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
path_clust = FIGURES_DIR / 'umap_hdbscan_clusters.png'
fig.savefig(path_clust, bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Sauvegardée : {path_clust}')

# ═════════════════════════════════════════════════════════════════════════════
# Figure 2 — Diagramme HR (annotations top-8 clusters, sans légende encombrée)
# ═════════════════════════════════════════════════════════════════════════════
teff_col, logg_col = 'teff_gspphot', 'logg_gspphot'

if teff_col in meta.columns and logg_col in meta.columns:
    teff_vals = meta[teff_col].values.astype(float)
    logg_vals = meta[logg_col].values.astype(float)
    valid_hr  = np.isfinite(teff_vals) & np.isfinite(logg_vals)

    # Compter les étoiles par cluster avec paramètres Gaia valides
    counts_hr = {cid: ((cluster_labels == cid) & valid_hr).sum() for cid in cluster_ids}
    top_clusters = sorted(counts_hr, key=counts_hr.get, reverse=True)[:8]

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=150)

    # ── Panneau gauche : HR coloré par cluster, annotations top-8 ──────────
    ax = axes[0]

    # Bruit d'abord (gris clair)
    noise_hr = noise_mask & valid_hr
    ax.scatter(
        teff_vals[noise_hr], logg_vals[noise_hr],
        c='lightgray', s=1, alpha=0.15, rasterized=True, linewidths=0, zorder=1,
    )

    # Tous les clusters (petit, sans légende)
    for cid in cluster_ids:
        m = (cluster_labels == cid) & valid_hr
        ax.scatter(
            teff_vals[m], logg_vals[m],
            c=[color_map[cid]], s=3, alpha=0.65,
            rasterized=True, linewidths=0, zorder=2,
        )

    # Annotations sur les 8 plus grands clusters dans l'espace HR
    for cid in top_clusters:
        m = (cluster_labels == cid) & valid_hr
        n_hr = m.sum()
        if n_hr < 20:
            continue
        med_teff = np.median(teff_vals[m])
        med_logg = np.median(logg_vals[m])
        ax.text(
            med_teff, med_logg, f"C{cid}\nn={n_hr}",
            fontsize=7.5, fontweight='bold', ha='center', va='center',
            color='white',
            bbox=dict(boxstyle='round,pad=0.25', fc=color_map[cid],
                      ec='none', alpha=0.90),
            zorder=5,
        )

    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.set_xlabel('$T_{eff}$ (K)', fontsize=12)
    ax.set_ylabel('log g (dex)', fontsize=11)
    ax.set_title('Diagramme HR — clusters HDBSCAN (top-8 annotés)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.2)

    # Colorbar discrète
    sm2 = plt.cm.ScalarMappable(
        cmap=cmap_clusters,
        norm=mcolors.Normalize(vmin=0, vmax=n_clusters - 1),
    )
    sm2.set_array([])
    cb2 = plt.colorbar(sm2, ax=ax, fraction=0.035, pad=0.02, label='Cluster ID')
    cb2.ax.tick_params(labelsize=8)

    # ── Panneau droit : UMAP coloré par T_eff ──────────────────────────────
    ax = axes[1]
    valid_teff = np.isfinite(teff_vals)
    sc = ax.scatter(
        Z_umap[valid_teff, 0], Z_umap[valid_teff, 1],
        c=teff_vals[valid_teff], cmap='plasma',
        vmin=np.nanpercentile(teff_vals[valid_teff], 2),
        vmax=np.nanpercentile(teff_vals[valid_teff], 98),
        s=2, alpha=0.55, rasterized=True, linewidths=0,
    )
    plt.colorbar(sc, ax=ax, label='T_eff (K)', fraction=0.046, pad=0.04)
    ax.set_xlabel('UMAP axe 1', fontsize=11)
    ax.set_title('UMAP colore par T_eff', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.2)

    fig.suptitle(
        'HDBSCAN — Localisation dans le diagramme HR · top-8 clusters annotés\n'
        'LAMOST DR5 × Gaia DR3',
        fontsize=14, fontweight='bold',
    )
    plt.tight_layout()
    path_hr = FIGURES_DIR / 'umap_hdbscan_hr.png'
    fig.savefig(path_hr, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'✓ Sauvegardée : {path_hr}')

# ═════════════════════════════════════════════════════════════════════════════
# Figure 3 — Sensibilité à min_cluster_size
# ═════════════════════════════════════════════════════════════════════════════
min_sizes    = [50, 100, 150, 300, 500]
sens_results = []
for mcs in min_sizes:
    cl = hdbscan.HDBSCAN(
        min_cluster_size=mcs, min_samples=HDBSCAN_MIN_SAMPLES,
        metric='euclidean', cluster_selection_method='eom',
    ).fit_predict(Z_umap)
    nc = len(set(cl)) - (1 if -1 in cl else 0)
    nn = (cl == -1).sum()
    sens_results.append({'min_cluster_size': mcs, 'n_clusters': nc,
                         'n_noise': nn, 'pct_noise': 100 * nn / len(cl)})

df_sens = pd.DataFrame(sens_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
axes[0].plot(df_sens['min_cluster_size'], df_sens['n_clusters'],
             'o-', color='#1A759F', lw=2, ms=7)
axes[0].axvline(HDBSCAN_MIN_CLUSTER, color='#D4690A', ls='--', lw=1.5,
                label=f'Valeur analyse fine ({HDBSCAN_MIN_CLUSTER})')
axes[0].axvline(300, color='#2D6A4F', ls=':', lw=1.5,
                label='Valeur présentation (300)')
axes[0].set_xlabel('min_cluster_size', fontsize=11)
axes[0].set_ylabel('Nombre de clusters', fontsize=11)
axes[0].set_title('Clusters détectés vs min_cluster_size', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_sens['min_cluster_size'], df_sens['pct_noise'],
             's-', color='#134074', lw=2, ms=7)
axes[1].axvline(HDBSCAN_MIN_CLUSTER, color='#D4690A', ls='--', lw=1.5,
                label=f'Valeur analyse fine ({HDBSCAN_MIN_CLUSTER})')
axes[1].axvline(300, color='#2D6A4F', ls=':', lw=1.5,
                label='Valeur présentation (300)')
axes[1].set_xlabel('min_cluster_size', fontsize=11)
axes[1].set_ylabel('% de points classés bruit', fontsize=11)
axes[1].set_title('Points bruit vs min_cluster_size', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Sensibilité HDBSCAN — impact de min_cluster_size\n'
             'Deux régimes : analyse fine (75) vs présentation (300)',
             fontsize=13)
plt.tight_layout()
path_sens = FIGURES_DIR / 'umap_hdbscan_sensitivity.png'
fig.savefig(path_sens, bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Sauvegardée : {path_sens}')
print()
print('Sensibilité HDBSCAN :')
print(df_sens.to_string(index=False))

# ═════════════════════════════════════════════════════════════════════════════
# Figure 4 — Version PRÉSENTATION (min_cluster_size=300, ~13 clusters lisibles)
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Calcul version présentation (min_cluster_size=300)…")
clusterer_pres = hdbscan.HDBSCAN(
    min_cluster_size=300, min_samples=HDBSCAN_MIN_SAMPLES,
    metric='euclidean', cluster_selection_method='eom',
)
labels_pres = clusterer_pres.fit_predict(Z_umap)
ids_pres    = sorted(set(labels_pres) - {-1})
nc_pres     = len(ids_pres)
nn_pres     = (labels_pres == -1).sum()
print(f"  {nc_pres} clusters · {nn_pres} points bruit ({100*nn_pres/len(labels_pres):.1f}%)")

# Palette distincte pour version présentation
cmap_pres  = plt.get_cmap('tab20', nc_pres)
cmap_p     = {cid: cmap_pres(i) for i, cid in enumerate(ids_pres)}
cmap_p[-1] = (0.82, 0.82, 0.82, 0.25)
noise_pres = labels_pres == -1

fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=150)

# ── Panneau gauche : UMAP version présentation ──────────────────────────────
ax = axes[0]
ax.scatter(
    Z_umap[noise_pres, 0], Z_umap[noise_pres, 1],
    c='lightgray', s=1.5, alpha=0.3,
    rasterized=True, linewidths=0, zorder=1,
    label=f'Bruit (n={nn_pres})',
)
for cid in ids_pres:
    mask = labels_pres == cid
    ax.scatter(
        Z_umap[mask, 0], Z_umap[mask, 1],
        c=[cmap_p[cid]], s=3, alpha=0.80,
        rasterized=True, linewidths=0, zorder=2,
        label=f'C{cid} (n={mask.sum()})',
    )
    # Annotation centroïde
    cx, cy = centroid(Z_umap, mask)
    ax.text(cx, cy, f'C{cid}', fontsize=9, fontweight='bold',
            ha='center', va='center', color='white',
            bbox=dict(boxstyle='round,pad=0.2', fc=cmap_p[cid], ec='none', alpha=0.9),
            zorder=5)

ax.legend(markerscale=4, fontsize=9, framealpha=0.9,
          loc='lower right', ncol=2)
ax.set_xlabel('UMAP axe 1', fontsize=12)
ax.set_ylabel('UMAP axe 2', fontsize=12)
ax.set_title(f'HDBSCAN — {nc_pres} populations stellaires', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2)

# ── Panneau droit : HR version présentation ─────────────────────────────────
ax = axes[1]
if teff_col in meta.columns and logg_col in meta.columns:
    ax.scatter(
        teff_vals[noise_pres & valid_hr], logg_vals[noise_pres & valid_hr],
        c='lightgray', s=1, alpha=0.15, rasterized=True, linewidths=0, zorder=1,
    )
    for cid in ids_pres:
        m = (labels_pres == cid) & valid_hr
        n_hr = m.sum()
        ax.scatter(
            teff_vals[m], logg_vals[m],
            c=[cmap_p[cid]], s=4, alpha=0.75,
            rasterized=True, linewidths=0, zorder=2,
        )
        if n_hr >= 30:
            med_t = np.median(teff_vals[m])
            med_g = np.median(logg_vals[m])
            ax.text(med_t, med_g, f'C{cid}', fontsize=9, fontweight='bold',
                    ha='center', va='center', color='white',
                    bbox=dict(boxstyle='round,pad=0.2', fc=cmap_p[cid], ec='none', alpha=0.9),
                    zorder=5)
    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.set_xlabel('T_eff (K)', fontsize=12)
    ax.set_ylabel('log g (dex)', fontsize=12)
    ax.set_title('Diagramme HR — populations HDBSCAN', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.2)

fig.suptitle(
    f'Découverte automatique de populations stellaires — HDBSCAN (min_size=300)\n'
    f'{nc_pres} populations · {nn_pres} anomalies ({100*nn_pres/len(labels_pres):.1f}%) · '
    'LAMOST DR5 × Gaia DR3',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
path_pres = FIGURES_DIR / 'umap_hdbscan_present.png'
fig.savefig(path_pres, bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Sauvegardée : {path_pres}')

In [ ]:
# ── Profil physique des populations découvertes par HDBSCAN ─────────────
# Variables Gaia disponibles dans meta
PHYS_COLS = [
    ('teff_gspphot',   'T_eff (K)'),
    ('logg_gspphot',   'log g (dex)'),
    ('mh_gspphot',     '[Fe/H]'),
    ('bp_rp',          'G_BP-G_RP'),
    ('phot_g_mean_mag','G mag'),
    ('distance_gspphot','Distance (pc)'),
]

# Construire un DataFrame complet avec cluster + meta
df_profile = meta.copy().reset_index(drop=True)
df_profile['cluster']     = cluster_labels
df_profile['class_lamost'] = y

# ── Tableau 1 : paramètres physiques moyens par cluster ──────────────────
available_cols = [(col, label) for col, label in PHYS_COLS if col in df_profile.columns]
col_names      = [col for col, _ in available_cols]
col_labels     = [label for _, label in available_cols]

# Exclure le bruit (-1) pour les moyennes
df_clusters = df_profile[df_profile['cluster'] != -1].copy()
# ── Profil physique des populations découvertes par HDBSCAN ─────────────
# Variables Gaia disponibles dans meta
PHYS_COLS = [
    ('teff_gspphot',   'T_eff (K)'),
    ('logg_gspphot',   'log g (dex)'),
    ('mh_gspphot',     '[Fe/H]'),
    ('bp_rp',          'G_BP-G_RP'),
    ('phot_g_mean_mag','G mag'),
    ('distance_gspphot','Distance (pc)'),
]

# Construire un DataFrame complet avec cluster + meta
df_profile = meta.copy().reset_index(drop=True)
df_profile['cluster']     = cluster_labels
df_profile['class_lamost'] = y

# ── Tableau 1 : paramètres physiques moyens par cluster ──────────────────
available_cols = [(col, label) for col, label in PHYS_COLS if col in df_profile.columns]
col_names      = [col for col, _ in available_cols]
col_labels     = [label for _, label in available_cols]

# Exclure le bruit (-1) pour les moyennes
df_clusters = df_profile[df_profile['cluster'] != -1].copy()

summary = df_clusters.groupby('cluster')[col_names].agg(['mean', 'std'])
summary.columns = ['_'.join(c) for c in summary.columns]

# Nombre de spectres par cluster
counts = df_clusters['cluster'].value_counts().sort_index()

# Classe dominante par cluster
dominant_class = df_clusters.groupby('cluster')['class_lamost'].apply(
    lambda x: x.value_counts().index[0]
).rename('Classe dominante')

# Assemblage du tableau final
import pandas as pd
rows = []
for cid in sorted(df_clusters['cluster'].unique()):
    row = {'Cluster': f'C{cid}', 'n spectres': int(counts.get(cid, 0)),
           'Classe dom.': dominant_class.get(cid, '?')}
    for col, label in available_cols:
        mean_col = f'{col}_mean'
        std_col  = f'{col}_std'
        if mean_col in summary.columns:
            mu  = summary.loc[cid, mean_col]
            sig = summary.loc[cid, std_col]
            row[label] = f'{mu:.0f} ± {sig:.0f}' if col == 'teff_gspphot' \
                         else f'{mu:.2f} ± {sig:.2f}'
        else:
            row[label] = 'N/A'
    rows.append(row)

# Ligne bruit
df_noise_row = df_profile[df_profile['cluster'] == -1]
row_noise = {'Cluster': 'Bruit', 'n spectres': len(df_noise_row),
             'Classe dom.': '—'}
for col, label in available_cols:
    if col in df_noise_row.columns:
        vals = df_noise_row[col].dropna()
        if len(vals) > 0:
            row_noise[label] = f'{vals.mean():.0f}' if col == 'teff_gspphot' \
                               else f'{vals.mean():.2f}'
        else:
            row_noise[label] = 'N/A'
rows.append(row_noise)

df_table = pd.DataFrame(rows).set_index('Cluster')

print('=' * 80)
print('  PROFIL PHYSIQUE DES POPULATIONS DÉCOUVERTES PAR HDBSCAN')
print('  (paramètres Gaia DR3, moyenne ± écart-type par cluster)')
print('=' * 80)
print(df_table.to_string())
print('=' * 80)

# ── Tableau 2 : interprétation astrophysique ─────────────────────────────
print('\nInterprétation astrophysique automatique :')
print('-' * 50)
for cid in sorted(df_clusters['cluster'].unique()):
    teff_mu = summary.loc[cid, 'teff_gspphot_mean'] \
              if 'teff_gspphot_mean' in summary.columns else None
    logg_mu = summary.loc[cid, 'logg_gspphot_mean'] \
              if 'logg_gspphot_mean' in summary.columns else None
    mh_mu   = summary.loc[cid, 'mh_gspphot_mean'] \
              if 'mh_gspphot_mean' in summary.columns else None

    n = int(counts.get(cid, 0))
    desc = []

    if teff_mu is not None and not pd.isna(teff_mu):
        if teff_mu > 7500:
            desc.append('étoiles chaudes (A–F)')
        elif teff_mu > 5500:
            desc.append('étoiles de type solaire (F–G)')
        elif teff_mu > 4000:
            desc.append('étoiles froides (G–K)')
        else:
            desc.append('étoiles très froides (K–M)')

    if logg_mu is not None and not pd.isna(logg_mu):
        if logg_mu < 2.5:
            desc.append('supergéantes')
        elif logg_mu < 3.5:
            desc.append('géantes')
        else:
            desc.append('naines (séquence principale)')

    if mh_mu is not None and not pd.isna(mh_mu):
        if mh_mu < -0.8:
            desc.append('[Fe/H] très faible — population vieille/halo')
        elif mh_mu < -0.3:
            desc.append('[Fe/H] faible — population II')
        else:
            desc.append('[Fe/H] solaire — population I')

    interpretation = ', '.join(desc) if desc else 'paramètres Gaia insuffisants'
    print(f'  C{cid:2d} (n={n:5d}) : {interpretation}')

print(f'  Bruit  (n={len(df_noise_row):5d}) : anomalies spectrales, objets atypiques')
summary = df_clusters.groupby('cluster')[col_names].agg(['mean', 'std'])
summary.columns = ['_'.join(c) for c in summary.columns]

# Nombre de spectres par cluster
counts = df_clusters['cluster'].value_counts().sort_index()

# Classe dominante par cluster
dominant_class = df_clusters.groupby('cluster')['class_lamost'].apply(
    lambda x: x.value_counts().index[0]
).rename('Classe dominante')

# Assemblage du tableau final
import pandas as pd
rows = []
for cid in sorted(df_clusters['cluster'].unique()):
    row = {'Cluster': f'C{cid}', 'n spectres': int(counts.get(cid, 0)),
           'Classe dom.': dominant_class.get(cid, '?')}
    for col, label in available_cols:
        mean_col = f'{col}_mean'
        std_col  = f'{col}_std'
        if mean_col in summary.columns:
            mu  = summary.loc[cid, mean_col]
            sig = summary.loc[cid, std_col]
            row[label] = f'{mu:.0f} ± {sig:.0f}' if col == 'teff_gspphot' \
                         else f'{mu:.2f} ± {sig:.2f}'
        else:
            row[label] = 'N/A'
    rows.append(row)

# Ligne bruit
df_noise_row = df_profile[df_profile['cluster'] == -1]
row_noise = {'Cluster': 'Bruit', 'n spectres': len(df_noise_row),
             'Classe dom.': '—'}
for col, label in available_cols:
    if col in df_noise_row.columns:
        vals = df_noise_row[col].dropna()
        if len(vals) > 0:
            row_noise[label] = f'{vals.mean():.0f}' if col == 'teff_gspphot' \
                               else f'{vals.mean():.2f}'
        else:
            row_noise[label] = 'N/A'
rows.append(row_noise)

df_table = pd.DataFrame(rows).set_index('Cluster')

print('=' * 80)
print('  PROFIL PHYSIQUE DES POPULATIONS DÉCOUVERTES PAR HDBSCAN')
print('  (paramètres Gaia DR3, moyenne ± écart-type par cluster)')
print('=' * 80)
print(df_table.to_string())
print('=' * 80)

# ── Tableau 2 : interprétation astrophysique ─────────────────────────────
print('\nInterprétation astrophysique automatique :')
print('-' * 50)
for cid in sorted(df_clusters['cluster'].unique()):
    teff_mu = summary.loc[cid, 'teff_gspphot_mean'] \
              if 'teff_gspphot_mean' in summary.columns else None
    logg_mu = summary.loc[cid, 'logg_gspphot_mean'] \
              if 'logg_gspphot_mean' in summary.columns else None
    mh_mu   = summary.loc[cid, 'mh_gspphot_mean'] \
              if 'mh_gspphot_mean' in summary.columns else None

    n = int(counts.get(cid, 0))
    desc = []

    if teff_mu is not None and not pd.isna(teff_mu):
        if teff_mu > 7500:
            desc.append('étoiles chaudes (A–F)')
        elif teff_mu > 5500:
            desc.append('étoiles de type solaire (F–G)')
        elif teff_mu > 4000:
            desc.append('étoiles froides (G–K)')
        else:
            desc.append('étoiles très froides (K–M)')

    if logg_mu is not None and not pd.isna(logg_mu):
        if logg_mu < 2.5:
            desc.append('supergéantes')
        elif logg_mu < 3.5:
            desc.append('géantes')
        else:
            desc.append('naines (séquence principale)')

    if mh_mu is not None and not pd.isna(mh_mu):
        if mh_mu < -0.8:
            desc.append('[Fe/H] très faible — population vieille/halo')
        elif mh_mu < -0.3:
            desc.append('[Fe/H] faible — population II')
        else:
            desc.append('[Fe/H] solaire — population I')

    interpretation = ', '.join(desc) if desc else 'paramètres Gaia insuffisants'
    print(f'  C{cid:2d} (n={n:5d}) : {interpretation}')

print(f'  Bruit  (n={len(df_noise_row):5d}) : anomalies spectrales, objets atypiques')

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #0F4C5C;">Interprétation — HDBSCAN et découverte non supervisée de populations stellaires</h4>

<p style="margin: 0 0 8px 0;">
<strong>Ce que l'algorithme accomplit.</strong> En n'utilisant que la <em>géométrie 2D de l'espace UMAP</em> — sans aucun label, sans paramètre physique, sans connaissance astrophysique préalable — HDBSCAN identifie automatiquement des groupes distincts. Lorsqu'on croise ces groupes avec les paramètres Gaia DR3, on redécouvre les grandes familles de la séquence stellaire : naines de la séquence principale, géantes rouges, étoiles de type solaire, populations métaliquement distinctes.
</p>

<p style="margin: 0 0 8px 0;">
<strong>Le bruit comme information.</strong> Les points classés comme "bruit" par HDBSCAN ne sont pas des erreurs — ce sont les objets <em>spectroscopiquement atypiques</em> : spectres de basse qualité, binaires à raies doubles, étoiles variables, ou les rares galaxies et QSOs du catalogue. L'autoencodeur (NB03) les détecte via une MSE de reconstruction élevée ; HDBSCAN les détecte via leur isolement topologique dans l'espace UMAP. Les deux approches se confirment mutuellement.
</p>

<p style="margin: 0 0 8px 0;">
<strong>Comparaison avec XGBoost supervisé.</strong> Le pipeline AstroSpectro principal (XGBoost) atteint ~84% de balanced accuracy sur les types spectraux LAMOST. HDBSCAN, sans aucune supervision, retrouve des structures qui correspondent aux grandes classes physiques — ce qui valide que les features spectroscopiques engineerées contiennent bien le signal astrophysique nécessaire, et pas seulement du signal de classification arbitraire.
</p>

<p style="margin: 0;">
<strong>Sensibilité aux hyperparamètres.</strong> <code>min_cluster_size</code> contrôle la granularité : une valeur faible fragmente la séquence principale en sous-populations fines (branche des naines K, naines M), une valeur élevée ne retient que les grandes structures. La valeur choisie (150 sur 42 920 spectres, soit ~0.35%) offre un bon compromis entre découverte et robustesse.
</p>
</div>

<a id="hdbscan-spectres"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.3 · Profil spectroscopique moyen par cluster HDBSCAN</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<p style="margin: 0;">
HDBSCAN a identifié les clusters géométriquement dans l'espace UMAP. Cette section <em>ferme la boucle</em> : on retourne dans l'espace des features originales pour calculer le spectre moyen de chaque population. Si les clusters correspondent à des populations physiques réelles, leurs profils de features doivent être caractéristiques des types spectraux connus (A, F, G, K, M).
</p>
</div>

In [ ]:
# ── Chargement des features originales (même fichier que NB01) ──────────
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Importer les mêmes listes d'exclusion que DimRedDataLoader
from dimred.data_loader import (
    _GAIA_META_COLS,
    _SNR_COLS,
    _INSTRUMENTAL_COLS,
    _NON_SPECTRO_PREFIXES,
)

# Le features_stem est stocké dans pca_output (chargé en cellule setup)
features_csv = Path(paths['PROCESSED_DIR']) / f'{features_stem}.csv'

if not features_csv.exists():
    print(f'⚠  Fichier features introuvable : {features_csv}')
    print('   → §1.4 sera ignorée. Vérifier paths["PROCESSED_DIR"].')
    _FEATURES_OK = False
else:
    _FEATURES_OK = True
    df_raw = pd.read_csv(features_csv, low_memory=False)
    print(f'Features chargées : {df_raw.shape}')

if _FEATURES_OK:
    # ── Appliquer le même filtre que DimRedDataLoader (SNR r >= 10) ──────
    if 'snr_r' in df_raw.columns:
        df_filt = df_raw[df_raw['snr_r'] >= 10.0].copy()
    else:
        df_filt = df_raw.copy()

    # Sélectionner uniquement les colonnes numériques features
    # (exclure colonnes meta exactement comme DimRedDataLoader)

    # ── Exclusions identiques à DimRedDataLoader(spectro_only=True) ──────
    EXCLUDE = set([
        'obsid', 'fits_name', 'filename_original', 'plan_id', 'mjd', 'jd',
        'designation', 'object_name', 'class', 'subclass', 'label', 'main_class',
        'author', 'data_version', 'date_creation', 'telescope',
        'fiber_type', 'catalog_object_type', 'magnitude_type',
        'heliocentric_correction', 'obs_date_utc', 'phot_variable_flag',
        'source_id', 'gaia_ra', 'gaia_dec',
    ])
    EXCLUDE |= set(_GAIA_META_COLS)
    EXCLUDE |= set(_SNR_COLS)
    EXCLUDE |= set(_INSTRUMENTAL_COLS)   # ← ra, dec, redshift, is_good_ruwe, etc.

    feat_cols = [
        c for c in df_filt.columns
        if c not in EXCLUDE
        and not c.startswith(_NON_SPECTRO_PREFIXES)   # ← filtre préfixes Gaia
        and pd.api.types.is_numeric_dtype(df_filt[c])
        and df_filt[c].nunique() > 1
        and df_filt[c].isna().mean() <= 0.10
    ]

    df_feat = df_filt[feat_cols].dropna().reset_index(drop=True)
    print(f'Matrice features filtrée : {df_feat.shape}')

    # Vérification de la cohérence avec Z_umap
    if len(df_feat) != len(Z_umap):
        print(f'⚠  Taille incohérente : features={len(df_feat)}, Z_umap={len(Z_umap)}')
        print('   Troncature au minimum commun pour alignement.')
        n_common = min(len(df_feat), len(Z_umap))
        df_feat        = df_feat.iloc[:n_common]
        cluster_labels_aligned = cluster_labels[:n_common]
    else:
        cluster_labels_aligned = cluster_labels
        print(f'✓ Alignement parfait : {len(df_feat)} spectres')

    # ── Calcul du profil moyen par cluster ──────────────────────────────
    from sklearn.preprocessing import StandardScaler

    scaler  = StandardScaler()
    X_std   = scaler.fit_transform(df_feat.values.astype(float))
    X_std_df = pd.DataFrame(X_std, columns=feat_cols)
    X_std_df['cluster'] = cluster_labels_aligned

    # Moyenne standardisée par cluster (excluant le bruit -1)
    cluster_means = (
        X_std_df[X_std_df['cluster'] != -1]
        .groupby('cluster')[feat_cols]
        .mean()
    )
    print(f'Profils moyens calculés : {cluster_means.shape} (clusters × features)')

    # ── Top features discriminantes (variance inter-cluster maximale) ───
    inter_var   = cluster_means.var(axis=0)
    top30_feats = inter_var.nlargest(30).index.tolist()

    # ── Figure 1 : Heatmap clusters × top-30 features ────────────────────
    fig, ax = plt.subplots(
        figsize=(18, max(6, n_clusters * 0.5 + 2)), dpi=150
    )

    heatmap_data = cluster_means[top30_feats]
    im = ax.imshow(
        heatmap_data.values,
        aspect='auto', cmap='RdBu_r',
        vmin=-2.5, vmax=2.5,
    )
    plt.colorbar(im, ax=ax, label='Score standardisé moyen', fraction=0.02, pad=0.01)

    ax.set_xticks(range(len(top30_feats)))
    ax.set_xticklabels(
        [f.replace('feature_','').replace('_eq_width','_EW')[:18]
         for f in top30_feats],
        rotation=45, ha='right', fontsize=7.5,
    )

    # Étiquettes Y : cluster ID + taille + interprétation physique rapide
    ytick_labels = []
    for cid in cluster_means.index:
        n_c = (cluster_labels_aligned == cid).sum()
        ytick_labels.append(f'C{cid}  (n={n_c})')
    ax.set_yticks(range(len(cluster_means)))
    ax.set_yticklabels(ytick_labels, fontsize=8)
    ax.set_xlabel('Top-30 features discriminantes (variance inter-cluster)', fontsize=11)
    ax.set_ylabel('Cluster HDBSCAN', fontsize=11)
    ax.set_title(
        'Profil spectroscopique moyen par cluster HDBSCAN\n'
        '(scores standardisés — rouge = excès, bleu = déficit)',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    path_hm = FIGURES_DIR / 'umap_hdbscan_feature_heatmap.png'
    fig.savefig(path_hm, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'✓ Sauvegardée : {path_hm}')

    # ── Figure 2 : Top-5 features caractéristiques de chaque cluster ────
    # Pour chaque cluster, trouver les 5 features où il s'écarte le plus
    # de la moyenne globale (score absolu le plus élevé)
    n_show = min(n_clusters, 12)  # Afficher au plus 12 clusters
    top_cids = sorted(
        cluster_means.index,
        key=lambda c: (cluster_labels_aligned == c).sum(),
        reverse=True
    )[:n_show]

    ncols = 4
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), dpi=120)
    axes_flat = axes.flatten() if n_show > 1 else [axes]

    # Palette (réutilise color_map défini en cellule 10)
    for ax_i, cid in zip(axes_flat, top_cids):
        profile = cluster_means.loc[cid]
        top5    = profile.abs().nlargest(5).index.tolist()
        vals    = profile[top5].values
        colors  = ['#C0392B' if v > 0 else '#2980B9' for v in vals]
        labels  = [
            f.replace('feature_','').replace('_eq_width','\nEW')[:14]
            for f in top5
        ]
        bars = ax_i.barh(range(5), vals, color=colors, edgecolor='none')
        ax_i.set_yticks(range(5))
        ax_i.set_yticklabels(labels, fontsize=8)
        ax_i.axvline(0, color='gray', lw=0.8, ls='--')
        n_c = (cluster_labels_aligned == cid).sum()
        ax_i.set_title(
            f'C{cid}  (n={n_c})',
            fontsize=10, fontweight='bold',
            color=color_map.get(cid, '#333'),
        )
        ax_i.set_xlabel('Score moyen (σ)', fontsize=8)
        ax_i.grid(axis='x', alpha=0.3)
        ax_i.set_xlim(-3, 3)

    # Masquer les axes vides
    for ax_i in axes_flat[n_show:]:
        ax_i.axis('off')

    fig.suptitle(
        'Top-5 features caractéristiques par cluster HDBSCAN\n'
        '(rouge = excès vs population globale, bleu = déficit)',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    path_bar = FIGURES_DIR / 'umap_hdbscan_feature_profiles.png'
    fig.savefig(path_bar, bbox_inches='tight', dpi=120)
    plt.show()
    print(f'✓ Sauvegardée : {path_bar}')
    print()
    print('Top-5 features discriminantes par cluster (hors bruit) :')
    for cid in sorted(cluster_means.index):
        top5 = cluster_means.loc[cid].abs().nlargest(5).index.tolist()
        vals = cluster_means.loc[cid, top5].round(2).tolist()
        pairs = ', '.join(f'{f.replace("feature_","")[:15]}={v:+.2f}' for f, v in zip(top5, vals))
        print(f'  C{cid:2d} : {pairs}')

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #0F4C5C;">Interprétation — Profils spectraux et classification MK</h4>
<p style="margin: 0 0 8px 0;">
La heatmap ferme la boucle entre la réduction de dimension et la physique stellaire. Les clusters montrant un <strong>excès de Hα_eq_width</strong> et un <strong>déficit de Mg_b / CaII</strong> correspondent aux étoiles chaudes (types A–F) — exactement l'opposition que PC1 de la PCA captait avec ρ=+0.832. Les clusters inverses (CaII fort, Hα faible) correspondent aux étoiles froides K–M.
</p>
<p style="margin: 0;">
Ce résultat montre que la classification non supervisée (UMAP → HDBSCAN) <em>redécouvre automatiquement</em> la séquence de classification spectrale MK établie au 19ème siècle — mais à partir de rien d'autre que la géométrie de 208 descripteurs spectraux. C'est la validation physique ultime du pipeline.
</p>
</div>

<a id="xgboost-umap"></a>

<div style="background: linear-gradient(135deg, #134074 0%, #3C6E71 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(19,64,116,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.4 · UMAP × XGBoost — Le pont supervisé / non-supervisé</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(19,64,116,0.12), rgba(60,110,113,0.12));
    border-left: 6px solid #134074;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<p style="margin: 0 0 8px 0;">
Le pipeline AstroSpectro (supervisé, XGBoost, ~84% balanced accuracy) et l'analyse de réduction de dimension (non supervisée, PHY-3500) ont jusqu'ici fonctionné en <em>silo</em>. Cette cellule crée le lien manquant : on applique le classificateur XGBoost entraîné aux mêmes 42 920 spectres et on colorie l'espace UMAP par les <strong>prédictions du modèle</strong>.
</p>
<p style="margin: 0;">
Questions clés : Les classes prédites par XGBoost correspondent-elles aux clusters HDBSCAN ? Où se situent les erreurs de classification (confusion F/G) dans l'espace UMAP ? Les zones d'incertitude sont-elles aux frontières entre clusters ?
</p>
</div>

In [ ]:
# ── Chargement du SpectralClassifier XGBoost ────────────────────────────
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from pathlib import Path

sys.path.insert(0, str(Path('../..').resolve() / 'src'))
from utils import latest_file

# Chercher le fichier .pkl du modèle dans MODELS_DIR
MODEL_PATH = latest_file(paths['MODELS_DIR'], 'spectral_classifier*.pkl')

if MODEL_PATH is None:
    print('⚠  Aucun fichier spectral_classifier*.pkl trouvé dans :', paths['MODELS_DIR'])
    print('   → §1.5 sera ignorée.')
    _XGB_OK = False
else:
    _XGB_OK = True
    print(f'Modèle XGBoost trouvé : {Path(MODEL_PATH).name}')

if _XGB_OK:
    try:
        from pipeline.classifier import SpectralClassifier
    except ImportError:
        from classifier import SpectralClassifier

    clf = SpectralClassifier.load_model(MODEL_PATH)
    print(f'Classes connues        : {clf.class_labels}')
    print(f'Features utilisées     : {len(clf.feature_names_used)}')
    print(f'Features sélectionnées : {len(clf.selected_features_ or [])}')

# ── Chargement et alignement du CSV de features ──────────────────────────
# CORRECTION BUG 1 : le DimRedDataLoader applique SNR + dropna sur les
# colonnes PCA features. Il faut reproduire les DEUX filtres pour obtenir
# exactement les 42 920 lignes alignées avec Z_umap.

if _XGB_OK:
    features_csv = Path(paths['PROCESSED_DIR']) / f'{features_stem}.csv'
    if not features_csv.exists():
        features_csv_str = latest_file(paths['PROCESSED_DIR'], 'features_*.csv')
        if features_csv_str is None:
            print('⚠  Aucun features_*.csv trouvé.')
            _XGB_OK = False
        else:
            features_csv = Path(features_csv_str)

if _XGB_OK:
    df_raw_xgb = pd.read_csv(features_csv, low_memory=False)

    # Filtre 1 : SNR
    if 'snr_r' in df_raw_xgb.columns:
        df_f = df_raw_xgb[df_raw_xgb['snr_r'] >= 10.0].copy()
    else:
        df_f = df_raw_xgb.copy()

    # Filtre 2 : reproduire le dropna de DimRedDataLoader
    # (exactement les mêmes colonnes exclues que dans _select_feature_columns)
    _EXCL = {
        'obsid','fits_name','filename_original','plan_id','mjd','jd',
        'designation','object_name','class','subclass','author','data_version',
        'date_creation','telescope','fiber_type','catalog_object_type',
        'magnitude_type','heliocentric_correction','radial_velocity_corr',
        'obs_date_utc','phot_variable_flag','source_id','gaia_ra','gaia_dec',
        'teff_gspphot','logg_gspphot','mh_gspphot','bp_rp','bp_g','g_rp',
        'phot_g_mean_mag','distance_gspphot','pmra','pmdec','parallax','ruwe',
        'ag_gspphot','ebpminrp_gspphot',
        'snr_u','snr_g','snr_r','snr_i','snr_z',
    }
    pca_feat_cols = [
        c for c in df_f.columns
        if c not in _EXCL
        and pd.api.types.is_numeric_dtype(df_f[c])
        and df_f[c].nunique() > 1
        and df_f[c].isna().mean() <= 0.10
    ]
    # Appliquer le même dropna que DimRedDataLoader
    df_aligned = df_f.dropna(subset=pca_feat_cols).reset_index(drop=True)
    print(f'Lignes après filtre SNR + dropna : {len(df_aligned)}  (Z_umap : {len(Z_umap)})')

    if len(df_aligned) != len(Z_umap):
        print(f'⚠  Toujours désaligné — troncature au minimum commun.')
        n_common = min(len(df_aligned), len(Z_umap))
        df_aligned     = df_aligned.iloc[:n_common]
        Z_umap_aligned = Z_umap[:n_common]
        y_aligned      = y[:n_common]
        cl_aligned     = cluster_labels[:n_common]
    else:
        Z_umap_aligned = Z_umap
        y_aligned      = y
        cl_aligned     = cluster_labels
        print(f'✓ Alignement parfait : {len(df_aligned)} spectres')

    # Vérifier les features nécessaires
    needed  = clf.feature_names_used
    missing = [f for f in needed if f not in df_aligned.columns]
    if missing:
        pct = len(missing) / len(needed)
        print(f'⚠  {len(missing)}/{len(needed)} features manquantes ({pct:.0%})')
        if pct > 0.30:
            print('   Trop de features manquantes — §1.5 ignorée.')
            _XGB_OK = False
        else:
            print(f'   Imputation par 0 pour les features manquantes.')
            for f in missing:
                df_aligned[f] = 0.0

# ── Prédiction ───────────────────────────────────────────────────────────
# CORRECTION BUG 2 : passer un DataFrame (pas un array numpy)
# Le ColumnTransformer sklearn a été fitté sur un DataFrame avec noms
# de colonnes — il lève TypeError si on lui passe un array sans noms.

if _XGB_OK:
    # DataFrame dans le bon ordre de colonnes (pas de .values !)
    X_pred_df = df_aligned[needed].fillna(0)

    print(f'\nPrédiction sur {len(X_pred_df)} spectres…')
    y_pred_enc = clf.model_pipeline.predict(X_pred_df)

    if clf.label_encoder is not None:
        y_pred = clf.label_encoder.inverse_transform(y_pred_enc)
    else:
        y_pred = np.array(y_pred_enc, dtype=str)

    # Probabilités de confiance
    try:
        proba      = clf.model_pipeline.predict_proba(X_pred_df)
        confidence = proba.max(axis=1)
        _HAS_PROBA = True
    except Exception:
        confidence = np.ones(len(y_pred))
        _HAS_PROBA = False

    classes_pred = sorted(set(y_pred))
    print(f'Classes prédites : {dict(zip(*np.unique(y_pred, return_counts=True)))}')

    # ── Palette couleurs stellaires ───────────────────────────────────
    STELLAR_COLORS = {
        'O': '#9B59B6', 'B': '#3498DB', 'A': '#1ABC9C',
        'F': '#F1C40F', 'G': '#E67E22', 'K': '#E74C3C', 'M': '#922B21',
        'C': '#884EA0', 'W': '#17202A', 's': '#7F8C8D',
        'STAR': '#4C72B0', 'GALAXY': '#DD8452', 'QSO': '#55A868',
    }

    # ── Figure : UMAP 1×3 (XGBoost | Confiance | HDBSCAN) ────────────
    fig, axes = plt.subplots(1, 3, figsize=(22, 8), dpi=150)

    ax = axes[0]
    for cls in classes_pred:
        mask = y_pred == cls
        ax.scatter(
            Z_umap_aligned[mask, 0], Z_umap_aligned[mask, 1],
            c=[STELLAR_COLORS.get(cls, '#888888')],
            s=2, alpha=0.55, linewidths=0, rasterized=True,
            label=f'{cls} (n={mask.sum()})',
        )
    ax.legend(markerscale=4, fontsize=9, framealpha=0.9, loc='best')
    ax.set_title('Prédictions XGBoost', fontsize=12, fontweight='bold')
    ax.set_xlabel('UMAP axe 1', fontsize=10)
    ax.set_ylabel('UMAP axe 2', fontsize=10)
    ax.grid(True, alpha=0.2)

    ax = axes[1]
    if _HAS_PROBA:
        sc = ax.scatter(
            Z_umap_aligned[:, 0], Z_umap_aligned[:, 1],
            c=confidence, cmap='RdYlGn',
            vmin=0.4, vmax=1.0,
            s=2, alpha=0.60, linewidths=0, rasterized=True,
        )
        plt.colorbar(sc, ax=ax, label='Confiance max (prob)', fraction=0.046, pad=0.04)
        ax.set_title('Confiance XGBoost (P_max)', fontsize=12, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'predict_proba\nnon disponible',
                transform=ax.transAxes, ha='center', va='center', fontsize=12)
        ax.set_title('Confiance (indisponible)', fontsize=12)
    ax.set_xlabel('UMAP axe 1', fontsize=10)
    ax.grid(True, alpha=0.2)

    ax = axes[2]
    noise_m = cl_aligned == -1
    ax.scatter(
        Z_umap_aligned[noise_m, 0], Z_umap_aligned[noise_m, 1],
        c='lightgray', s=1.5, alpha=0.25, rasterized=True, linewidths=0,
    )
    for cid in sorted(set(cl_aligned) - {-1}):
        m = cl_aligned == cid
        ax.scatter(
            Z_umap_aligned[m, 0], Z_umap_aligned[m, 1],
            c=[color_map.get(cid, '#888')], s=2, alpha=0.65,
            rasterized=True, linewidths=0,
        )
    ax.set_title('Clusters HDBSCAN (référence)', fontsize=12, fontweight='bold')
    ax.set_xlabel('UMAP axe 1', fontsize=10)
    ax.grid(True, alpha=0.2)

    fig.suptitle(
        'UMAP × XGBoost — Supervisé vs Non-supervisé\nLAMOST DR5 × Gaia DR3',
        fontsize=14, fontweight='bold',
    )
    plt.tight_layout()
    path_xgb = FIGURES_DIR / 'umap_xgboost_predictions.png'
    fig.savefig(path_xgb, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'✓ Sauvegardée : {path_xgb}')

    # ── Correspondance XGBoost ↔ HDBSCAN ─────────────────────────────
    print('\n── Correspondance XGBoost ↔ HDBSCAN (top-3 classes par cluster) ──')
    df_cross = pd.DataFrame({'xgb_pred': y_pred, 'cluster': cl_aligned})
    for cid in sorted(set(cl_aligned)):
        label  = 'Bruit' if cid == -1 else f'C{cid:2d}'
        subset = df_cross[df_cross['cluster'] == cid]
        top3   = subset['xgb_pred'].value_counts().head(3)
        top3_s = ', '.join(f'{cls}={n}' for cls, n in top3.items())
        print(f'  {label} (n={len(subset):5d}) : {top3_s}')

    # ── Zone de confusion F/G ─────────────────────────────────────────
    fg_mask = np.isin(y_pred, ['F', 'G'])
    if fg_mask.sum() > 10:
        fig, ax = plt.subplots(figsize=(9, 8), dpi=150)
        ax.scatter(
            Z_umap_aligned[:, 0], Z_umap_aligned[:, 1],
            c='#EEEEEE', s=1, alpha=0.3, rasterized=True, linewidths=0,
        )
        for cls, col in [('F', '#F1C40F'), ('G', '#E67E22')]:
            m = y_pred == cls
            ax.scatter(
                Z_umap_aligned[m, 0], Z_umap_aligned[m, 1],
                c=col, s=3, alpha=0.7, linewidths=0,
                rasterized=True, label=f'Prédit {cls} (n={m.sum()})',
            )
        ax.legend(markerscale=3, fontsize=11, framealpha=0.9)
        ax.set_xlabel('UMAP axe 1', fontsize=11)
        ax.set_ylabel('UMAP axe 2', fontsize=11)
        ax.set_title(
            'Zone de confusion F/G — où XGBoost hésite-t-il ?\n'
            'LAMOST DR5 × Gaia DR3',
            fontsize=12, fontweight='bold',
        )
        ax.grid(True, alpha=0.25)
        plt.tight_layout()
        path_fg = FIGURES_DIR / 'umap_xgboost_FG_confusion.png'
        fig.savefig(path_fg, bbox_inches='tight', dpi=150)
        plt.show()
        print(f'✓ Sauvegardée : {path_fg}')

<div style="
    background: linear-gradient(135deg, rgba(19,64,116,0.12), rgba(60,110,113,0.12));
    border-left: 6px solid #134074;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #134074;">Interprétation — Correspondance supervisé / non-supervisé</h4>
<p style="margin: 0 0 8px 0;">
Si les clusters HDBSCAN et les prédictions XGBoost concordent largement, cela démontre que l'espace spectral contient un signal astrophysique robuste que deux approches fondamentalement différentes capturent de façon cohérente.
</p>
<p style="margin: 0 0 8px 0;">
La carte de confiance est particulièrement révélatrice : les zones de faible confiance (P_max &lt; 0.6) se concentrent aux <em>frontières entre clusters</em> HDBSCAN — là où la géométrie non-supervisée est elle-même ambiguë. Le modèle supervisé et l'algorithme non-supervisé s'accordent donc sur <em>où</em> les données sont ambiguës.
</p>
<p style="margin: 0;">
La confusion F/G est particulièrement instructive : ces étoiles occupent une région continue dans l'espace UMAP (pas de frontière nette), ce qui explique pourquoi XGBoost les confond — la séparation physique entre F et G est progressive, pas discrète.
</p>
</div>

<a id="umap3d"></a>

<div style="background: linear-gradient(135deg, #3C6E71 0%, #2D6A4F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(60,110,113,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.5 · Variété stellaire en 3D — UMAP interactif (Plotly)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(60,110,113,0.12), rgba(45,106,79,0.12));
    border-left: 6px solid #3C6E71;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<p style="margin: 0 0 8px 0;">
UMAP 2D projette les 50 dimensions PCA en 2, mais compresse nécessairement de l'information. Une projection en <strong>3 dimensions</strong> permet de vérifier si le troisième axe encode un paramètre physique indépendant de T_eff et log g — typiquement la <em>métallicité [Fe/H]</em> ou la <em>vitesse de rotation</em>.
</p>
<p style="margin: 0;">
La figure Plotly est sauvegardée en <code>.html</code> — ouvrez-la dans un navigateur pour faire tourner la variété stellaire en 3D lors de la présentation au Musée de la Civilisation.
</p>
</div>

In [ ]:
# ── Vérifier si Plotly est disponible ───────────────────────────────────
try:
    import plotly.express as px
    import plotly.graph_objects as go
    _PLOTLY_OK = True
except ImportError:
    print('⚠  Plotly non installé → pip install plotly')
    _PLOTLY_OK = False

if not _PLOTLY_OK:
    raise RuntimeError('Installer plotly avant de continuer : pip install plotly')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ── Calcul UMAP 3D ───────────────────────────────────────────────────────
print('Calcul UMAP 3D (n_components=3)…')
umap_3d_engine = EmbeddingEngine(
    method='umap',
    n_components=3,
    random_state=RANDOM_STATE,
    n_neighbors=15,
    min_dist=0.1,
)
Z_umap3 = umap_3d_engine.fit_transform(scores_95)
print(f'UMAP 3D terminé en {umap_3d_engine.fit_time_:.1f}s | shape : {Z_umap3.shape}')

# ── Analyse scientifique : que encode l'axe 3 ? ──────────────────────────
phys_params = [
    ('teff_gspphot',    'T_eff (K)'),
    ('logg_gspphot',    'log g'),
    ('mh_gspphot',      '[Fe/H]'),
    ('bp_rp',           'G_BP-G_RP'),
    ('phot_g_mean_mag', 'G mag'),
    ('parallax',        'Parallaxe'),
]

print('\nCorrélations Spearman : axes UMAP 3D ↔ paramètres Gaia')
print(f'{"Paramètre":<18} {"Axe 1":>10} {"Axe 2":>10} {"Axe 3":>10}')
print('-' * 52)
axis3_best_corr = 0
axis3_best_param = '?'
for col, label in phys_params:
    if col not in meta.columns:
        continue
    vals  = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    if valid.sum() < 100:
        continue
    r1, _ = spearmanr(Z_umap3[valid, 0], vals[valid])
    r2, _ = spearmanr(Z_umap3[valid, 1], vals[valid])
    r3, _ = spearmanr(Z_umap3[valid, 2], vals[valid])
    print(f'{label:<18} {r1:>+10.3f} {r2:>+10.3f} {r3:>+10.3f}')
    if abs(r3) > abs(axis3_best_corr):
        axis3_best_corr  = r3
        axis3_best_param = label
print('-' * 52)
print(f'Axe 3 corrèle principalement avec : {axis3_best_param} (ρ={axis3_best_corr:+.3f})')

# ── Construire le DataFrame Plotly ────────────────────────────────────────
df_3d = pd.DataFrame({
    'UMAP1': Z_umap3[:, 0],
    'UMAP2': Z_umap3[:, 1],
    'UMAP3': Z_umap3[:, 2],
    'Classe': y,
})

# Ajouter les paramètres Gaia disponibles
for col, label in phys_params:
    if col in meta.columns:
        df_3d[label] = meta[col].values

# Cluster HDBSCAN (string pour catégoriel dans Plotly)
df_3d['Cluster'] = [
    'Bruit' if c == -1 else f'C{c}'
    for c in cluster_labels
]

# ── Figure 1 : coloré par classe LAMOST ──────────────────────────────────
print('\nGénération de la figure Plotly 3D (colorée par classe)…')

# Sous-échantillonnage pour Plotly (max 20k points pour réactivité)
N_PLOT = min(20_000, len(df_3d))
rng    = np.random.default_rng(RANDOM_STATE)
idx_s  = rng.choice(len(df_3d), size=N_PLOT, replace=False)
df_plot = df_3d.iloc[idx_s].copy()
print(f'Points affichés : {N_PLOT} / {len(df_3d)} (sous-échantillonnage pour réactivité)')

COLOR_MAP_LAMOST = {
    'STAR': '#4C72B0', 'GALAXY': '#DD8452', 'QSO': '#55A868', 'Unknown': '#999999',
}

fig_3d_class = px.scatter_3d(
    df_plot,
    x='UMAP1', y='UMAP2', z='UMAP3',
    color='Classe',
    color_discrete_map=COLOR_MAP_LAMOST,
    opacity=0.6,
    size_max=3,
    title=f'Variété stellaire UMAP 3D — Classification LAMOST DR5<br>'
          f'(n={N_PLOT} spectres · axes corrélés avec : '
          f'axe3↔{axis3_best_param} ρ={axis3_best_corr:+.2f})',
    labels={'UMAP1': 'UMAP axe 1', 'UMAP2': 'UMAP axe 2', 'UMAP3': 'UMAP axe 3'},
)
fig_3d_class.update_traces(marker=dict(size=2))
fig_3d_class.update_layout(
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title=f'UMAP 3  (↔ {axis3_best_param})',
    ),
    legend=dict(itemsizing='constant'),
    margin=dict(l=0, r=0, b=0, t=50),
)

# Sauvegarder HTML
path_3d_html = FIGURES_DIR / 'umap3d_classes.html'
fig_3d_class.write_html(str(path_3d_html))
print(f'✓ Figure interactive sauvegardée : {path_3d_html}')
fig_3d_class.show()

# ── Figure 2 : coloré par T_eff ──────────────────────────────────────────
if 'T_eff (K)' in df_plot.columns:
    fig_3d_teff = px.scatter_3d(
        df_plot.dropna(subset=['T_eff (K)']),
        x='UMAP1', y='UMAP2', z='UMAP3',
        color='T_eff (K)',
        color_continuous_scale='plasma',
        range_color=[
            float(df_plot['T_eff (K)'].quantile(0.02)),
            float(df_plot['T_eff (K)'].quantile(0.98)),
        ],
        opacity=0.6,
        title='Variété stellaire UMAP 3D — T_eff (K) · LAMOST DR5 × Gaia DR3',
    )
    fig_3d_teff.update_traces(marker=dict(size=2))
    fig_3d_teff.update_layout(
        scene=dict(
            xaxis_title='UMAP 1',
            yaxis_title='UMAP 2',
            zaxis_title=f'UMAP 3  (↔ {axis3_best_param})',
        ),
        margin=dict(l=0, r=0, b=0, t=50),
    )
    path_3d_teff = FIGURES_DIR / 'umap3d_teff.html'
    fig_3d_teff.write_html(str(path_3d_teff))
    print(f'✓ Figure T_eff sauvegardée : {path_3d_teff}')
    fig_3d_teff.show()

# ── Figure 3 : coloré par [Fe/H] ─────────────────────────────────────────
if '[Fe/H]' in df_plot.columns:
    fig_3d_feh = px.scatter_3d(
        df_plot.dropna(subset=['[Fe/H]']),
        x='UMAP1', y='UMAP2', z='UMAP3',
        color='[Fe/H]',
        color_continuous_scale='RdYlBu',
        range_color=[
            float(df_plot['[Fe/H]'].quantile(0.02)),
            float(df_plot['[Fe/H]'].quantile(0.98)),
        ],
        opacity=0.6,
        title='Variété stellaire UMAP 3D — [Fe/H] · LAMOST DR5 × Gaia DR3',
    )
    fig_3d_feh.update_traces(marker=dict(size=2))
    fig_3d_feh.update_layout(
        scene=dict(
            xaxis_title='UMAP 1',
            yaxis_title='UMAP 2',
            zaxis_title=f'UMAP 3  (↔ {axis3_best_param})',
        ),
        margin=dict(l=0, r=0, b=0, t=50),
    )
    path_3d_feh = FIGURES_DIR / 'umap3d_feh.html'
    fig_3d_feh.write_html(str(path_3d_feh))
    print(f'✓ Figure [Fe/H] sauvegardée : {path_3d_feh}')
    fig_3d_feh.show()

# ── Résumé axe 3 ─────────────────────────────────────────────────────────
print()
print('=' * 55)
print('  RÉSUMÉ — UMAP 3D')
print('=' * 55)
print(f'  Temps de calcul     : {umap_3d_engine.fit_time_:.1f} s')
print(f'  Axe 3 encode        : {axis3_best_param} (ρ={axis3_best_corr:+.3f})')
print(f'  HTML exportés vers  : {FIGURES_DIR}')
print(f'    umap3d_classes.html')
print(f'    umap3d_teff.html')
print(f'    umap3d_feh.html')
print('  → Ouvrir dans un navigateur pour la présentation interactive')
print('=' * 55)

<div style="
    background: linear-gradient(135deg, rgba(60,110,113,0.12), rgba(45,106,79,0.12));
    border-left: 6px solid #3C6E71;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #3C6E71;">Interprétation — Ce qu'encode le 3ème axe UMAP</h4>
<p style="margin: 0 0 8px 0;">
Le tableau de corrélations Spearman révèle si le passage de 2D à 3D ajoute de l'information physique réelle ou seulement du bruit. Si l'axe 3 corrèle fortement avec [Fe/H] (et peu avec T_eff déjà capturée par les axes 1-2), cela démontre que l'espace spectral stellaire a au moins <strong>3 dimensions physiques indépendantes</strong> : température, gravité de surface, et abondance métallique.
</p>
<p style="margin: 0 0 8px 0;">
Ce résultat serait en accord avec la classification spectrale MK étendue qui distingue les populations galactiques (Pop I, Pop II, halo) par leur métallicité, et avec la trouvaille SHAP d'AstroSpectro où Ca II H&K et Mg b (indicateurs de métallicité) dominent la classification XGBoost.
</p>
<p style="margin: 0;">
<strong>Pour la présentation au Musée :</strong> faire tourner le globe 3D tout en expliquant que chaque axe correspond à un paramètre physique fondamental de l'étoile — l'audience verra littéralement la structure de l'espace stellaire en trois dimensions.
</p>
</div>

<a id="tsne"></a>

<div style="background: linear-gradient(135deg, #134B5F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">2 · Embedding t-SNE</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(19,75,95,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<strong>Principe :</strong> t-SNE transforme les distances en probabilités de voisinage et minimise une divergence de Kullback-Leibler entre l’espace d’entrée et l’espace projeté.
</div>

En haute dimension, les similarités conditionnelles sont définies par :

$$
p_{j|i} = \frac{\exp\left(-\|x_i-x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k\neq i}\exp\left(-\|x_i-x_k\|^2 / 2\sigma_i^2\right)}
$$

La projection 2D est optimisée via :

$$
\mathrm{KL}(P\|Q)=\sum_{i\neq j} p_{ij}\log\left(\frac{p_{ij}}{q_{ij}}\right)
$$

- `perplexity` règle le compromis local/global (ordre de grandeur du nombre de voisins effectifs).
- t-SNE est excellent pour la séparation locale, mais ses distances globales sont moins interprétables.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
tsne_engine = EmbeddingEngine(
    method="tsne",
    n_components=2,
    random_state=RANDOM_STATE,
    perplexity=30,
    max_iter=1000,
)

Z_tsne = tsne_engine.fit_transform(scores_95)
print(f"t-SNE terminé en {tsne_engine.fit_time_:.1f}s")

In [ ]:
fig, axes = viz.plot_embedding_grid(
    Z_tsne, y=y, meta=meta,
    method="t-SNE",
    params=["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"],
    s=2, alpha=0.4,
    save_path=FIGURES_DIR / "tsne_grid.png",
)
plt.show()

<div style="
    background: linear-gradient(135deg, rgba(19,75,95,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #134B5F;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Interprétation - Lecture de la carte t-SNE</h4>
<p style="margin: 0 0 8px 0;">t-SNE sépare généralement mieux les voisinages immédiats que UMAP, ce qui le rend utile pour isoler de petits groupes et des structures fines.</p>
<p style="margin: 0;">En contrepartie, les distances entre amas ne doivent pas être interprétées comme des distances physiques globales fiables.</p>
</div>

- À privilégier pour l’exploration locale et la visualisation de sous-groupes.
- À compléter par UMAP/PCA pour une lecture plus globale de la géométrie des données.

<a id="stability"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">3 · Analyse de stabilité (Procrustes)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(26,117,159,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
La stabilité est évaluée en relançant chaque méthode avec plusieurs seeds, puis en alignant les cartes obtenues par analyse de Procrustes.
</div>

Après translation, rotation et mise à l’échelle optimales, la distance de Procrustes peut se résumer par :

$$
d_{\mathrm{proc}}(A,B)=\|\hat{A}-\hat{B}\|_F
$$

- Plus $d_{proc}$ est proche de 0, plus la méthode est stable face au hasard d’initialisation.
- Cette métrique est essentielle pour juger la robustesse scientifique d’un embedding.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
print("Calcul stabilité UMAP (5 seeds)...")
stab_umap = umap_engine.stability_report(scores_95, n_seeds=5)
print(stab_umap.to_string(index=False))

fig, ax = viz.plot_stability(
    stab_umap,
    save_path=FIGURES_DIR / "stability_umap.png",
)
plt.show()

<a id="trustworthiness"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">3.bis · Trustworthiness — fidélité de voisinage UMAP et t-SNE</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">La <strong>trustworthiness</strong> (Venna &amp; Oja, 2006) mesure dans quelle proportion les voisins proches dans l'espace d'entrée sont aussi proches dans l'espace réduit. Elle complète la stabilité Procrustes : où la stabilité évalue la <em>reproductibilité</em> (même résultat à chaque seed), la trustworthiness évalue la <em>fidélité</em> (les proximités sont-elles respectées).</p>
<p style="margin: 0;">Formellement, pour N points et k voisins :</p>
</div>

$$T(k) = 1 - \frac{2}{Nk(2N-3k-1)} \sum_{i=1}^{N} \sum_{j \in \mathcal{U}_k(i)} (r(i,j) - k)$$

où $\mathcal{U}_k(i)$ est l'ensemble des points qui sont parmi les $k$ plus proches voisins de $i$ dans l'espace 2D mais **pas** dans l'espace d'entrée, et $r(i,j)$ est leur rang dans l'espace d'entrée.

Un score proche de **1.0** = les voisinages sont parfaitement préservés.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #1A759F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Trustworthiness UMAP vs t-SNE ────────────────────────────────────────
from sklearn.manifold import trustworthiness
import numpy as np
import matplotlib.pyplot as plt
import warnings

print("Calcul de la trustworthiness (patience — O(N²) pour grands k)...")

# Sous-échantillonnage pour la vitesse (trustworthiness est O(N²))
N_TRUST = min(5000, len(scores_95))
rng_trust = np.random.default_rng(42)
idx_trust = rng_trust.choice(len(scores_95), size=N_TRUST, replace=False)

X_sub    = scores_95[idx_trust]
Zu_sub   = Z_umap[idx_trust]
Zt_sub   = Z_tsne[idx_trust]

K_VALUES = [5, 10, 15, 20, 30, 50, 100]

trust_umap = []
trust_tsne = []

for k in K_VALUES:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        tu = trustworthiness(X_sub, Zu_sub, n_neighbors=k, metric="euclidean")
        tt = trustworthiness(X_sub, Zt_sub, n_neighbors=k, metric="euclidean")
    trust_umap.append(tu)
    trust_tsne.append(tt)
    print(f"  k={k:3d} | UMAP : {tu:.4f} | t-SNE : {tt:.4f}")

print()
print(f"Trustworthiness moyenne UMAP  : {np.mean(trust_umap):.4f}")
print(f"Trustworthiness moyenne t-SNE : {np.mean(trust_tsne):.4f}")
print(f"(calculée sur N={N_TRUST} spectres sous-échantillonnés)")

# ── Figure ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

ax.plot(K_VALUES, trust_umap, "o-", color="#1A759F", lw=2.2,
        markersize=7, label=f"UMAP (moy. = {np.mean(trust_umap):.4f})")
ax.plot(K_VALUES, trust_tsne, "s--", color="#E8593C", lw=2.2,
        markersize=7, label=f"t-SNE (moy. = {np.mean(trust_tsne):.4f})")

ax.axhline(1.0, color="gray", lw=0.8, ls=":", alpha=0.6)
ax.fill_between(K_VALUES, trust_umap, trust_tsne,
                alpha=0.08, color="#7F8FA6")

ax.set_xlabel("Nombre de voisins k", fontsize=11)
ax.set_ylabel("Trustworthiness T(k)", fontsize=11)
ax.set_title(
    "Fidélité de voisinage — UMAP vs t-SNE\n"
    f"LAMOST DR5 · {N_TRUST} spectres · entrée : {scores_95.shape[1]} PCs",
    fontsize=12, fontweight="bold",
)
ax.set_ylim(0.75, 1.01)
ax.set_xlim(0, max(K_VALUES) * 1.05)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
plt.tight_layout()

save_path_trust = FIGURES_DIR / "umap_trustworthiness.png"
fig.savefig(save_path_trust, bbox_inches="tight", dpi=150)
plt.show()
print(f"✓ Sauvegardée : {save_path_trust}")

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #0F4C5C;">Interprétation — Trustworthiness et choix de méthode</h4>
<p style="margin: 0 0 8px 0;">Un score élevé à <strong>petit k</strong> signifie que les voisins immédiats sont fidèlement représentés — la structure locale est préservée. Un score élevé à <strong>grand k</strong> indique que la structure globale (groupes distants) est aussi respectée.</p>
<p style="margin: 0 0 8px 0;">En général, <strong>t-SNE</strong> excelle à petit k (structure locale très fine) mais dégrade souvent à grand k. <strong>UMAP</strong> offre un meilleur compromis local/global grâce à son initialisation spectrale et son optimisation du graphe de voisinage.</p>
<p style="margin: 0;">Combiné avec la stabilité Procrustes (t-SNE plus stable, UMAP plus reproductible dans la topologie globale), ces deux métriques donnent un portrait complet et objectif des deux méthodes.</p>
</div>

In [ ]:
print("Calcul stabilité t-SNE (5 seeds)...")
stab_tsne = tsne_engine.stability_report(scores_95, n_seeds=5)
print(stab_tsne.to_string(index=False))

fig, ax = viz.plot_stability(
    stab_tsne,
    save_path=FIGURES_DIR / "stability_tsne.png",
)
plt.show()

<a id="negative"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">4 · Contrôle négatif (données permutées)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Le contrôle négatif consiste à permuter les colonnes (ou les lignes selon le protocole) pour briser les corrélations réelles tout en conservant les distributions marginales.
</div>

Interprétation attendue :
- Si la structure observée sur les données réelles disparaît après permutation, cela appuie la validité physique de l’embedding.
- Si une structure similaire persiste, il faut suspecter un artefact méthodologique.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
Z_umap_neg = umap_engine.negative_control(scores_95)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)
classes = np.unique(y)

for ax, Z_plot, title in zip(
    axes,
    [Z_umap, Z_umap_neg],
    ["UMAP — Données réelles", "UMAP — Contrôle négatif (X permuté)"],
):
    for cls in classes:
        mask = y == cls
        ax.scatter(
            Z_plot[mask, 0], Z_plot[mask, 1],
            c=CLASS_COLORS.get(cls, "gray"),
            marker=CLASS_MARKERS.get(cls, "o"),
            s=2, alpha=0.4, label=cls, rasterized=True,
        )
    ax.set_title(title, fontsize=11)
    ax.legend(markerscale=3, fontsize=8)
    ax.set_aspect("equal", "box")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

fig.suptitle("Validation : données réelles vs contrôle négatif", fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "umap_negative_control.png", bbox_inches="tight", dpi=150)
plt.show()

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(26,117,159,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Interprétation - Contrôle négatif</h4>
<p style="margin: 0 0 8px 0;">Le contrôle négatif est une vérification critique : la permutation détruit la structure corrélée tout en conservant les distributions marginales.</p>
<p style="margin: 0;">La disparition des filaments sur les données permutées confirme que les motifs observés sur les données réelles proviennent de corrélations physiques et non d’un artefact de réduction de dimension.</p>
</div>

### Point méthode
- Cette étape joue un rôle de test de falsification.
- Elle renforce la crédibilité des interprétations astrophysiques tirées des projections UMAP/t-SNE.

<a id="sensitivity"></a>

<div style="background: linear-gradient(135deg, #134B5F 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">5 · Sensibilité aux hyperparamètres</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(19,75,95,0.12));
    border-left: 6px solid #134B5F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Cette section teste la robustesse qualitative des cartes face aux paramètres clés : `n_neighbors` (UMAP) et `perplexity` (t-SNE).
</div>

### Lecture scientifique
- Une méthode robuste conserve les grands motifs d’organisation quand on fait varier les hyperparamètres dans une plage raisonnable.
- Des changements trop brutaux signalent une dépendance excessive au réglage et limitent la reproductibilité.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #1A759F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# UMAP : variation de n_neighbors
sensitivity_umap = umap_engine.parameter_sensitivity(
    scores_95,
    param_grid={"n_neighbors": [5, 15, 30, 50, 100]},
)

fig, axes = plt.subplots(1, len(sensitivity_umap),
                         figsize=(5 * len(sensitivity_umap), 5), dpi=120)

for ax, (label, Z_s) in zip(axes, sensitivity_umap.items()):
    for cls in classes:
        mask = y == cls
        ax.scatter(Z_s[mask, 0], Z_s[mask, 1],
                   c=CLASS_COLORS.get(cls, "gray"),
                   marker=CLASS_MARKERS.get(cls, "o"),
                   s=1.5, alpha=0.35, rasterized=True)
    ax.set_title(f"UMAP\n{label}", fontsize=10)
    ax.set_aspect("equal", "box")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Sensibilité UMAP — variation de n_neighbors", fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "umap_sensitivity_n_neighbors.png", bbox_inches="tight", dpi=120)
plt.show()

In [ ]:
# t-SNE : variation de perplexity
sensitivity_tsne = tsne_engine.parameter_sensitivity(
    scores_95,
    param_grid={"perplexity": [5, 15, 30, 50, 100]},
)

fig, axes = plt.subplots(1, len(sensitivity_tsne),
                         figsize=(5 * len(sensitivity_tsne), 5), dpi=120)

for ax, (label, Z_s) in zip(axes, sensitivity_tsne.items()):
    for cls in classes:
        mask = y == cls
        ax.scatter(Z_s[mask, 0], Z_s[mask, 1],
                   c=CLASS_COLORS.get(cls, "gray"),
                   marker=CLASS_MARKERS.get(cls, "o"),
                   s=1.5, alpha=0.35, rasterized=True)
    ax.set_title(f"t-SNE\n{label}", fontsize=10)
    ax.set_aspect("equal", "box")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Sensibilité t-SNE — variation de perplexity", fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "tsne_sensitivity_perplexity.png", bbox_inches="tight", dpi=120)
plt.show()

<a id="hr"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">6 · Diagramme HR coloré par coordonnée d’embedding</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(26,117,159,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Projeter les coordonnées d’embedding sur le diagramme HR permet de vérifier si les axes appris suivent des gradients astrophysiques plausibles (température, évolution stellaire, métallicité).
</div>

Rappel (forme simplifiée) pour la magnitude absolue en bande G :

$$
M_G \approx G + 5\log_{10}(\varpi_{\mathrm{mas}}) - 10
$$

- Une cohérence visuelle entre structures HR et structures UMAP/t-SNE renforce l’interprétation physique.
- Cette étape relie directement la géométrie mathématique des embeddings à la physique stellaire.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
fig, ax = viz.plot_hr_diagram_embedding(
    meta=meta, Z=Z_umap, component=0, method="UMAP",
    save_path=FIGURES_DIR / "hr_diagram_umap_ax1.png",
)
plt.show()

fig, ax = viz.plot_hr_diagram_embedding(
    meta=meta, Z=Z_umap, component=1, method="UMAP",
    save_path=FIGURES_DIR / "hr_diagram_umap_ax2.png",
)
plt.show()

fig, ax = viz.plot_hr_diagram_embedding(
    meta=meta, Z=Z_tsne, component=0, method="t-SNE",
    save_path=FIGURES_DIR / "hr_diagram_tsne_ax1.png",
)
plt.show()

<a id="save"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #134B5F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">7 · Sauvegarde des embeddings et rapports de run</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(19,75,95,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #134B5F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Cette cellule exporte les artefacts du run (joblib, JSON, TXT) avec horodatage et version latest pour assurer la traçabilité expérimentale.
</div>

### Pourquoi c’est important
- Reproductibilité : mêmes paramètres, mêmes entrées, mêmes sorties.
- Audit scientifique : suivi des temps, de la stabilité, des hyperparamètres et des figures générées.
- Réutilisation : les embeddings sont immédiatement disponibles pour NB03 et le rapport final.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import json
import joblib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def _g(name, default=None):
    return globals().get(name, default)


def _class_counts(labels):
    classes, counts = np.unique(labels, return_counts=True)
    return {str(cls): int(cnt) for cls, cnt in zip(classes, counts)}


def _safe_df_records(df, max_rows=None):
    if df is None:
        return None
    out = df.copy()
    if max_rows is not None:
        out = out.head(max_rows)
    out = out.where(pd.notna(out), None)
    return out.to_dict(orient="records")


def _json_default(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    raise TypeError(f"Type non serialisable: {type(obj)!r}")


def _shape_or_none(x):
    if x is None:
        return None
    try:
        return list(np.asarray(x).shape)
    except Exception:
        return None


def _embedding_stats(Z):
    if Z is None:
        return None
    arr = np.asarray(Z)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return {"shape": list(arr.shape)}
    return {
        "shape": list(arr.shape),
        "x_min": float(np.nanmin(arr[:, 0])),
        "x_max": float(np.nanmax(arr[:, 0])),
        "y_min": float(np.nanmin(arr[:, 1])),
        "y_max": float(np.nanmax(arr[:, 1])),
        "x_mean": float(np.nanmean(arr[:, 0])),
        "y_mean": float(np.nanmean(arr[:, 1])),
        "x_std": float(np.nanstd(arr[:, 0])),
        "y_std": float(np.nanstd(arr[:, 1])),
    }


def _embedding_stats_nd(Z, max_axes=3):
    if Z is None:
        return None
    arr = np.asarray(Z)
    if arr.ndim != 2:
        return {"shape": list(arr.shape)}

    out = {"shape": list(arr.shape)}
    n_axes = min(arr.shape[1], max_axes)
    for i in range(n_axes):
        axis_vals = arr[:, i]
        out[f"axis_{i + 1}_min"] = float(np.nanmin(axis_vals))
        out[f"axis_{i + 1}_max"] = float(np.nanmax(axis_vals))
        out[f"axis_{i + 1}_mean"] = float(np.nanmean(axis_vals))
        out[f"axis_{i + 1}_std"] = float(np.nanstd(axis_vals))
    return out


def _sensitivity_shapes(sensitivity_dict):
    if sensitivity_dict is None:
        return None
    out = {}
    for label, Z_s in sensitivity_dict.items():
        out[str(label)] = list(np.asarray(Z_s).shape)
    return out


def _numeric_means(df):
    if df is None or len(df) == 0:
        return None
    cols = df.select_dtypes(include=[np.number]).columns
    return {str(c): float(df[c].mean()) for c in cols}


def _fmt_seconds(value):
    if value is None:
        return "N/A"
    try:
        v = float(value)
    except (TypeError, ValueError):
        return "N/A"
    return f"{v:.2f}s" if np.isfinite(v) else "N/A"


def _vector_stats(arr):
    if arr is None:
        return None
    vals = np.asarray(arr, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None
    return {
        "n": int(vals.size),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "p05": float(np.percentile(vals, 5)),
        "p50": float(np.percentile(vals, 50)),
        "p95": float(np.percentile(vals, 95)),
    }


def _path_if_defined(var_name):
    if var_name not in globals():
        return None
    try:
        return str(Path(globals()[var_name]))
    except Exception:
        return str(globals()[var_name])


def _cluster_top_counts(labels, top_n=10):
    vals = np.asarray(labels)
    unique = np.unique(vals)
    counts = {str(int(c)): int((vals == c).sum()) for c in unique if c != -1}
    return dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:top_n])


def _top3_pred_by_cluster(cluster_arr, pred_arr):
    out = {}
    c = np.asarray(cluster_arr)
    p = np.asarray(pred_arr)
    for cid in np.unique(c):
        mask = c == cid
        vc = pd.Series(p[mask]).value_counts().head(3)
        key = "noise" if cid == -1 else str(int(cid))
        out[key] = {str(k): int(v) for k, v in vc.items()}
    return out


# ----------------------------------------------------------------------------
# Resolve paths and available variables (robust to partial runs)
# ----------------------------------------------------------------------------
paths_var = _g("paths", {})
if isinstance(paths_var, dict) and "REPORTS_DIR" in paths_var:
    reports_dir = Path(paths_var["REPORTS_DIR"])
else:
    reports_dir = Path.cwd() / "data" / "reports"
    print("Warning: paths['REPORTS_DIR'] absent, fallback to", reports_dir)

figures_dir = Path(_g("FIGURES_DIR", reports_dir / "figures" / "phy3500" / "unknown"))
figures_dir.mkdir(parents=True, exist_ok=True)

pca_output_path = _path_if_defined("PCA_OUTPUT")

scores_95 = _g("scores_95")
Z_umap = _g("Z_umap")
Z_tsne = _g("Z_tsne")
Z_umap3 = _g("Z_umap3")
y = _g("y")
meta = _g("meta")
umap_engine_obj = _g("umap_engine")
tsne_engine_obj = _g("tsne_engine")

if Z_umap is None and Z_tsne is None and Z_umap3 is None:
    raise RuntimeError(
        "Aucun embedding detecte (Z_umap/Z_tsne/Z_umap3). "
        "Executer au moins une section d'embedding avant la sauvegarde."
    )

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = reports_dir / "runs" / "phy3500_umap_tsne"
run_dir.mkdir(parents=True, exist_ok=True)

umap_fit_time = getattr(umap_engine_obj, "fit_time_", None) if umap_engine_obj is not None else None
tsne_fit_time = getattr(tsne_engine_obj, "fit_time_", None) if tsne_engine_obj is not None else None

labels_norm = None
if y is not None:
    labels_norm = np.char.upper(np.asarray(y).astype(str))
    labels_norm = np.char.strip(labels_norm)
    alias_map = {
        "GLAXAXY": "GALAXY",
        "GALAXIE": "GALAXY",
        "QS0": "QSO",
    }
    for bad, good in alias_map.items():
        labels_norm = np.where(labels_norm == bad, good, labels_norm)

# Optional section: zoom/pair summary from Cell 8
zoom_summary = None
if (
    labels_norm is not None
    and Z_umap is not None
    and "ZOOM_X" in globals()
    and "ZOOM_Y" in globals()
):
    try:
        zx_min, zx_max = map(float, ZOOM_X)
        zy_min, zy_max = map(float, ZOOM_Y)
        zoom_mask = (
            (Z_umap[:, 0] >= zx_min)
            & (Z_umap[:, 0] <= zx_max)
            & (Z_umap[:, 1] >= zy_min)
            & (Z_umap[:, 1] <= zy_max)
        )
        zoom_summary = {
            "zoom_x": [zx_min, zx_max],
            "zoom_y": [zy_min, zy_max],
            "n_points_total": int(zoom_mask.sum()),
            "counts_in_zoom": {
                "STAR": int(((labels_norm == "STAR") & zoom_mask).sum()),
                "GALAXY": int(((labels_norm == "GALAXY") & zoom_mask).sum()),
                "QSO": int(((labels_norm == "QSO") & zoom_mask).sum()),
            },
            "pair_figures_expected": {
                "umap_all_classes_pair.png": bool((figures_dir / "umap_all_classes_pair.png").exists()),
                "umap_no_star_pair.png": bool((figures_dir / "umap_no_star_pair.png").exists()),
            },
        }
    except Exception as exc:
        zoom_summary = {"error": str(exc)}

# Optional section: HDBSCAN summary
hdbscan_summary = None
if "cluster_labels" in globals():
    cl = np.asarray(cluster_labels)
    n_total = int(cl.size)
    n_noise = int((cl == -1).sum())
    n_clusters = int(len(np.unique(cl[cl != -1])))

    hdbscan_summary = {
        "n_points": n_total,
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "pct_noise": float(100.0 * n_noise / max(1, n_total)),
        "largest_clusters": _cluster_top_counts(cl, top_n=12),
    }

    if "clusterer" in globals():
        hdbscan_summary["params"] = {
            "min_cluster_size": getattr(clusterer, "min_cluster_size", None),
            "min_samples": getattr(clusterer, "min_samples", None),
            "metric": getattr(clusterer, "metric", None),
            "cluster_selection_method": getattr(clusterer, "cluster_selection_method", None),
        }

    if "labels_pres" in globals():
        lp = np.asarray(labels_pres)
        n_noise_pres = int((lp == -1).sum())
        hdbscan_summary["presentation_mode"] = {
            "n_clusters": int(len(np.unique(lp[lp != -1]))),
            "n_noise": n_noise_pres,
            "pct_noise": float(100.0 * n_noise_pres / max(1, lp.size)),
            "largest_clusters": _cluster_top_counts(lp, top_n=12),
        }

# Optional section: XGBoost bridge summary
xgboost_summary = None
if "y_pred" in globals():
    yp = np.asarray(y_pred).astype(str)

    xgboost_summary = {
        "n_predictions": int(yp.size),
        "pred_class_counts": _class_counts(yp),
        "model_path": _path_if_defined("MODEL_PATH"),
    }

    if "confidence" in globals():
        xgboost_summary["confidence_stats"] = _vector_stats(confidence)

    if "classes_pred" in globals():
        xgboost_summary["classes_pred_list"] = [str(c) for c in classes_pred]

    y_ref = None
    if "y_aligned" in globals() and len(np.asarray(y_aligned)) == len(yp):
        y_ref = np.asarray(y_aligned).astype(str)
    elif labels_norm is not None and len(labels_norm) == len(yp):
        y_ref = labels_norm

    if y_ref is not None:
        xgboost_summary["accuracy_vs_reference"] = float(np.mean(yp == y_ref))
        cm = pd.crosstab(
            pd.Series(y_ref, name="true"),
            pd.Series(yp, name="pred"),
        )
        xgboost_summary["confusion_matrix"] = {
            "index": [str(v) for v in cm.index.tolist()],
            "columns": [str(v) for v in cm.columns.tolist()],
            "values": cm.astype(int).values.tolist(),
        }

    if "cl_aligned" in globals() and len(np.asarray(cl_aligned)) == len(yp):
        xgboost_summary["top3_pred_by_cluster"] = _top3_pred_by_cluster(cl_aligned, yp)

    if "fg_mask" in globals():
        xgboost_summary["n_pred_F_or_G"] = int(np.asarray(fg_mask).sum())

# Optional section: UMAP 3D summary
umap3d_summary = None
if Z_umap3 is not None:
    umap3d_summary = {
        "embedding_stats": _embedding_stats_nd(Z_umap3, max_axes=3),
        "axis3_best_param": str(_g("axis3_best_param", "N/A")),
        "axis3_best_corr": float(_g("axis3_best_corr")) if _g("axis3_best_corr") is not None else None,
        "html_exports": {
            "umap3d_classes": _path_if_defined("path_3d_html"),
            "umap3d_teff": _path_if_defined("path_3d_teff"),
            "umap3d_feh": _path_if_defined("path_3d_feh"),
        },
    }

png_files = sorted(str(p) for p in figures_dir.glob("*.png"))
html_files = sorted(str(p) for p in figures_dir.glob("*.html"))

stability_umap = _g("stab_umap")
stability_tsne = _g("stab_tsne")
stability_umap_records = _safe_df_records(stability_umap.reset_index(drop=True)) if stability_umap is not None else None
stability_tsne_records = _safe_df_records(stability_tsne.reset_index(drop=True)) if stability_tsne is not None else None

embeddings_output = {
    "timestamp_utc": timestamp,
    "Z_umap": Z_umap,
    "Z_tsne": Z_tsne,
    "Z_umap3": Z_umap3,
    "y": y,
    "meta": meta,
    "cluster_labels": _g("cluster_labels"),
    "cluster_labels_presentation": _g("labels_pres"),
    "xgb_predictions": _g("y_pred"),
    "xgb_confidence": _g("confidence"),
    "stability_umap": stability_umap,
    "stability_tsne": stability_tsne,
    "umap_params": getattr(umap_engine_obj, "params_used", None) if umap_engine_obj is not None else None,
    "tsne_params": getattr(tsne_engine_obj, "params_used", None) if tsne_engine_obj is not None else None,
    "umap_fit_time_s": float(umap_fit_time) if umap_fit_time is not None else None,
    "tsne_fit_time_s": float(tsne_fit_time) if tsne_fit_time is not None else None,
    "Z_umap_negative_control": _g("Z_umap_neg"),
    "sensitivity_umap": _g("sensitivity_umap"),
    "sensitivity_tsne": _g("sensitivity_tsne"),
    "features_stem": _g("features_stem"),
    "zoom_x": list(_g("ZOOM_X")) if _g("ZOOM_X") is not None else None,
    "zoom_y": list(_g("ZOOM_Y")) if _g("ZOOM_Y") is not None else None,
}

save_path_legacy = reports_dir / "phy3500_embeddings.joblib"
save_path_run = run_dir / f"phy3500_embeddings_{timestamp}.joblib"
save_path_latest = run_dir / "phy3500_embeddings_latest.joblib"

joblib.dump(embeddings_output, save_path_legacy)
joblib.dump(embeddings_output, save_path_run)
joblib.dump(embeddings_output, save_path_latest)

report = {
    "run_timestamp_utc": timestamp,
    "notebook": "notebooks/dimred/phy3500_02_umap_tsne.ipynb",
    "captured_sections": {
        "has_scores_95": scores_95 is not None,
        "has_Z_umap": Z_umap is not None,
        "has_Z_tsne": Z_tsne is not None,
        "has_Z_umap3": Z_umap3 is not None,
        "has_labels": y is not None,
        "has_meta": meta is not None,
        "has_hdbscan": "cluster_labels" in globals(),
        "has_xgboost": "y_pred" in globals(),
    },
    "paths": {
        "pca_output_path": pca_output_path,
        "figures_dir": str(figures_dir),
        "embeddings_joblib_path_legacy": str(save_path_legacy),
        "embeddings_joblib_path_run": str(save_path_run),
        "embeddings_joblib_path_latest": str(save_path_latest),
        "run_dir": str(run_dir),
        "xgboost_prediction_figure": _path_if_defined("path_xgb"),
        "xgboost_fg_confusion_figure": _path_if_defined("path_fg"),
    },
    "shapes": {
        "scores_95": _shape_or_none(scores_95),
        "Z_umap": _shape_or_none(Z_umap),
        "Z_tsne": _shape_or_none(Z_tsne),
        "Z_umap3": _shape_or_none(Z_umap3),
        "meta": _shape_or_none(meta),
    },
    "class_counts": _class_counts(labels_norm) if labels_norm is not None else None,
    "timings_seconds": {
        "umap_fit_time": float(umap_fit_time) if umap_fit_time is not None else None,
        "tsne_fit_time": float(tsne_fit_time) if tsne_fit_time is not None else None,
    },
    "parameters": {
        "umap": getattr(umap_engine_obj, "params_used", None) if umap_engine_obj is not None else None,
        "tsne": getattr(tsne_engine_obj, "params_used", None) if tsne_engine_obj is not None else None,
        "zoom_x": list(_g("ZOOM_X")) if _g("ZOOM_X") is not None else None,
        "zoom_y": list(_g("ZOOM_Y")) if _g("ZOOM_Y") is not None else None,
    },
    "embedding_stats": {
        "umap": _embedding_stats(Z_umap),
        "tsne": _embedding_stats(Z_tsne),
        "umap3d": _embedding_stats_nd(Z_umap3, max_axes=3),
        "umap_negative_control": _embedding_stats(_g("Z_umap_neg")),
    },
    "stability": {
        "umap": stability_umap_records,
        "tsne": stability_tsne_records,
        "umap_numeric_means": _numeric_means(stability_umap),
        "tsne_numeric_means": _numeric_means(stability_tsne),
    },
    "sensitivity": {
        "umap_shapes": _sensitivity_shapes(_g("sensitivity_umap")),
        "tsne_shapes": _sensitivity_shapes(_g("sensitivity_tsne")),
    },
    "zoom_pair_summary": zoom_summary,
    "hdbscan_summary": hdbscan_summary,
    "xgboost_summary": xgboost_summary,
    "umap3d_summary": umap3d_summary,
    "artifacts": {
        "png_files": png_files,
        "html_files": html_files,
        "n_png": len(png_files),
        "n_html": len(html_files),
    },
}

json_path = run_dir / f"phy3500_umap_tsne_run_{timestamp}.json"
json_latest_path = run_dir / "phy3500_umap_tsne_run_latest.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=_json_default)

with open(json_latest_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=_json_default)

text_lines = [
    "AstroSpectro | PHY-3500 UMAP/t-SNE Run Report",
    "=" * 78,
    f"Timestamp (UTC)             : {timestamp}",
    f"Notebook                    : notebooks/dimred/phy3500_02_umap_tsne.ipynb",
    f"PCA input file              : {pca_output_path}",
    f"Figures directory           : {figures_dir}",
    "",
    "[Saved artifacts]",
    f"- Embeddings joblib (legacy): {save_path_legacy}",
    f"- Embeddings joblib (run)   : {save_path_run}",
    f"- Embeddings joblib (latest): {save_path_latest}",
    f"- JSON run report           : {json_path}",
    f"- JSON run latest           : {json_latest_path}",
    "",
    "[Captured sections]",
    f"- has_Z_umap                : {report['captured_sections']['has_Z_umap']}",
    f"- has_Z_tsne                : {report['captured_sections']['has_Z_tsne']}",
    f"- has_Z_umap3               : {report['captured_sections']['has_Z_umap3']}",
    f"- has_hdbscan               : {report['captured_sections']['has_hdbscan']}",
    f"- has_xgboost               : {report['captured_sections']['has_xgboost']}",
    "",
    "[Shapes]",
    f"- scores_95                 : {_shape_or_none(scores_95)}",
    f"- Z_umap                    : {_shape_or_none(Z_umap)}",
    f"- Z_tsne                    : {_shape_or_none(Z_tsne)}",
    f"- Z_umap3                   : {_shape_or_none(Z_umap3)}",
    f"- meta                      : {_shape_or_none(meta)}",
    "",
    "[Class counts]",
    f"- y                         : {_class_counts(labels_norm) if labels_norm is not None else 'N/A'}",
    "",
    "[Timing]",
    f"- UMAP fit                  : {_fmt_seconds(umap_fit_time)}",
    f"- t-SNE fit                 : {_fmt_seconds(tsne_fit_time)}",
    "",
]

if zoom_summary is not None:
    text_lines.extend([
        "[Zoom pair summary]",
        f"- zoom_x                    : {zoom_summary.get('zoom_x', 'N/A')}",
        f"- zoom_y                    : {zoom_summary.get('zoom_y', 'N/A')}",
        f"- n_points_total            : {zoom_summary.get('n_points_total', 'N/A')}",
        f"- counts_in_zoom            : {zoom_summary.get('counts_in_zoom', 'N/A')}",
        "",
    ])

if hdbscan_summary is not None:
    text_lines.extend([
        "[HDBSCAN summary]",
        f"- n_clusters                : {hdbscan_summary.get('n_clusters', 'N/A')}",
        f"- n_noise                   : {hdbscan_summary.get('n_noise', 'N/A')}",
        f"- pct_noise                 : {hdbscan_summary.get('pct_noise', 'N/A')}",
        f"- largest_clusters          : {hdbscan_summary.get('largest_clusters', 'N/A')}",
        "",
    ])

if xgboost_summary is not None:
    text_lines.extend([
        "[XGBoost summary]",
        f"- n_predictions             : {xgboost_summary.get('n_predictions', 'N/A')}",
        f"- pred_class_counts         : {xgboost_summary.get('pred_class_counts', 'N/A')}",
        f"- accuracy_vs_reference     : {xgboost_summary.get('accuracy_vs_reference', 'N/A')}",
        f"- confidence_stats          : {xgboost_summary.get('confidence_stats', 'N/A')}",
        f"- model_path                : {xgboost_summary.get('model_path', 'N/A')}",
        "",
    ])

if umap3d_summary is not None:
    text_lines.extend([
        "[UMAP 3D summary]",
        f"- axis3_best_param          : {umap3d_summary.get('axis3_best_param', 'N/A')}",
        f"- axis3_best_corr           : {umap3d_summary.get('axis3_best_corr', 'N/A')}",
        f"- html_exports              : {umap3d_summary.get('html_exports', 'N/A')}",
        "",
    ])

if stability_umap is not None:
    text_lines.extend([
        "Stabilite UMAP (table complete):",
        stability_umap.round(6).to_string(index=False),
        "",
    ])

if stability_tsne is not None:
    text_lines.extend([
        "Stabilite t-SNE (table complete):",
        stability_tsne.round(6).to_string(index=False),
        "",
    ])

if _g("sensitivity_umap") is not None:
    text_lines.extend([
        "Sensibilite UMAP (shape par configuration):",
        str(_sensitivity_shapes(_g("sensitivity_umap"))),
        "",
    ])

if _g("sensitivity_tsne") is not None:
    text_lines.extend([
        "Sensibilite t-SNE (shape par configuration):",
        str(_sensitivity_shapes(_g("sensitivity_tsne"))),
        "",
    ])

if _g("Z_umap_neg") is not None:
    text_lines.extend([
        "Controle negatif UMAP (stats):",
        str(_embedding_stats(_g("Z_umap_neg"))),
        "",
    ])

text_lines.extend([
    "[Artifacts inventory]",
    f"- PNG count                 : {len(png_files)}",
    f"- HTML count                : {len(html_files)}",
    "",
    "PNG files:",
])
text_lines.extend([f"  - {p}" for p in png_files])
text_lines.append("")
text_lines.append("HTML files:")
text_lines.extend([f"  - {p}" for p in html_files])
text_lines.append("")

text_path = run_dir / f"phy3500_umap_tsne_run_{timestamp}.txt"
text_latest_path = run_dir / "phy3500_umap_tsne_run_latest.txt"
text_content = "\n".join(text_lines)

with open(text_path, "w", encoding="utf-8") as f:
    f.write(text_content)

with open(text_latest_path, "w", encoding="utf-8") as f:
    f.write(text_content)

print(f"Embeddings sauvegardes      -> {save_path_legacy}")
print(f"Embeddings run (timestamp)  -> {save_path_run}")
print(f"Embeddings latest run       -> {save_path_latest}")
print(f"Rapport JSON run            -> {json_path}")
print(f"Rapport TXT run             -> {text_path}")
print(f"Raccourci JSON latest       -> {json_latest_path}")
print(f"Raccourci TXT latest        -> {text_latest_path}")
print(f"  Z_umap shape              : {_shape_or_none(Z_umap)}")
print(f"  Z_tsne shape              : {_shape_or_none(Z_tsne)}")
print(f"  Figures PNG inventoriees  : {len(png_files)}")
print(f"  Figures HTML inventoriees : {len(html_files)}")

<div style="
    background: linear-gradient(135deg, #102542 0%, #0F4C5C 50%, #1A759F 100%);
    padding: 18px 22px;
    border-radius: 12px;
    margin: 18px 0 12px 0;
    box-shadow: 0 8px 24px rgba(16,37,66,0.30);
">
  <h2 style="color: white; margin: 0; font-weight: 350; letter-spacing: 1px;">Résumé des résultats</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 12px;
">
Tableau complété à partir du run <strong>20260405T214233Z</strong>
(<code>data/reports/runs/phy3500_umap_tsne/phy3500_umap_tsne_run_20260405T214233Z.json</code>).
</div>

| Analyse | Résultat clé (run 20260405T214233Z) |
|---------|--------------------------------------|
| Échantillons utilisés | 43 019 objets (91 scores PCA, mode features) |
| Répartition des classes | STAR=42 956, GALAXY=56, QSO=7 |
| UMAP (n_neighbors=15, min_dist=0.1) | temps = 40.13 s ; shape = (43 019, 2) |
| t-SNE (perplexity=30) | temps = 80.20 s ; shape = (43 019, 2) |
| Stabilité moyenne UMAP (Procrustes) | 0.0239 (variabilité modérée entre seeds) |
| Stabilité moyenne t-SNE (Procrustes) | 0.000379 (très stable dans ce protocole) |
| Contrôle négatif UMAP | structure contractée (σx=2.200, σy=1.750) vs réel (σx=3.932, σy=4.541) |
| Sensibilité hyperparamètres | motifs globaux conservés pour UMAP et t-SNE sur les grilles testées |
| HDBSCAN (min_cluster_size=75) | 20 clusters ; 6.14% de bruit (2 643 / 43 019) |
| UMAP 3D (axe 3) | corrélation principale avec log g (ρ=-0.214) |

### Lecture rapide pour le rapport
- UMAP offre un excellent compromis entre lisibilité globale, gradients physiques et coût de calcul sur 43k objets.
- t-SNE fournit une séparation locale très fine, avec une stabilité seed-to-seed exceptionnellement élevée dans ce protocole.
- Le contrôle négatif confirme que les structures observées sur les données réelles portent une information corrélée astrophysique.
- Le clustering HDBSCAN met en évidence 20 populations avec une fraction de bruit limitée (~6%), cohérente avec une structure non supervisée riche.

<div style="
    margin-top: 14px;
    padding: 12px 14px;
    border-left: 5px solid #0F4C5C;
    background: linear-gradient(135deg, rgba(15,76,92,0.10), rgba(26,117,159,0.10));
    border-radius: 8px;
">
  <strong>Notebook suivant :</strong> <a href="./phy3500_03_autoencoder.ipynb">phy3500_03_autoencoder.ipynb</a>
</div>

<div style="
    background: linear-gradient(135deg, #134B5F 0%, #1A759F 100%);
    padding: 16px 20px;
    border-radius: 10px;
    margin: 18px 0 10px 0;
    box-shadow: 0 6px 18px rgba(19,75,95,0.25);
">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 0.8px;">Références scientifiques et techniques</h2>
</div>

### UMAP et apprentissage de variétés
1. McInnes, L., Healy, J., & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*. arXiv:1802.03426. (fondements théoriques et algorithmiques de UMAP)
2. Narayan, A., Berger, B., & Cho, H. (2021). *Assessing single-cell transcriptomic variability through density-preserving data visualization*. Nature Biotechnology, 39, 765-774. (discussion pratique sur les propriétés de visualisation de type UMAP)

### t-SNE et voisinages probabilistes
3. van der Maaten, L., & Hinton, G. (2008). *Visualizing Data using t-SNE*. Journal of Machine Learning Research, 9, 2579-2605. (article fondateur t-SNE)
4. van der Maaten, L. (2014). *Accelerating t-SNE using Tree-Based Algorithms*. JMLR, 15, 3221-3245. (aspects numériques et scalabilité)

### Stabilité, alignement et validation
5. Gower, J. C. (1975). *Generalized Procrustes Analysis*. Psychometrika, 40(1), 33-51. (base de l’alignement Procrustes)
6. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning* (chapitres sur l’évaluation et la robustesse des représentations). MIT Press. (cadre général sur stabilité des représentations)

### Données astrophysiques et contexte instrumental
7. Gaia Collaboration (2023). *Gaia Data Release 3: Summary of the content and survey properties*. Astronomy & Astrophysics, 674, A1. (paramètres astrométriques et photométriques utilisés)
8. Luo, A.-L., et al. (2019). *The LAMOST DR5 release*. Research in Astronomy and Astrophysics. (contexte instrumental LAMOST et produits spectraux)

### Implémentation et bonnes pratiques Python
9. scikit-learn developers. *TSNE documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html
10. umap-learn developers. *UMAP documentation*. https://umap-learn.readthedocs.io/

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-top: 12px;
">
Ces références couvrent la théorie des embeddings non linéaires, les méthodes de validation (stabilité/contrôle négatif), le contexte astrophysique (Gaia/LAMOST) et les détails d’implémentation.
</div>